# Native Language Identification using HuBERT Features - CPU Version

**Complete Implementation: Data Prep → MFCC Baseline → HuBERT Extraction → Layer Analysis → Deep Learning Models**

This notebook contains the full pipeline for Native Language Identification of Indian English speakers using:
- MFCC features (baseline)
- HuBERT features (13 layers from facebook/hubert-large-ls960-ft)
- Traditional ML: Random Forest with hyperparameter tuning
- Deep Learning: CNN, BiLSTM, Transformer architectures

## Table of Contents
1. Setup & Data Preparation
2. MFCC Feature Extraction & Baseline RF Model
3. HuBERT Feature Extraction (All 13 Layers)
4. Layer-wise Analysis & Best Layer Selection
5. Deep Learning Models (CNN, BiLSTM, Transformer)
6. Comprehensive Experiments & Comparison
7. Results Summary

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive

 --71---97---103---97---110---32---82---101---100---100---121-gaganreddy001-gmail-com-2980360a-872.pdf
'Apex-(NLP-HuBert)'
 Assignment-1.zip
'CLV Documentation (1).gdoc'
'Colab Notebooks'
'Copy of Another copy of jupyter.ipynb'
 GaganReddy-HITAM-2024-BATCH-certificate.pdf
'Innofiesta fee payment  (1).pdf'
'Innofiesta fee payment .pdf'
 NLI_HuBERT_Proj-2025
 NLI_HuBERT_Project
 NLI_HuBERT_Project-20251111T063720Z-1-002
 NOTEBOOK_TO_SCRIPTS.md
'Offer letter .pdf'
 PROJECT_OVERVIEW.md
 PROJECT_REPORT.md
 SUBMISSION_READY.md


In [3]:
!ls /content/drive/MyDrive/NLI_HuBERT_Project

 data			  metadata.csv	     'nli-2(GPU).ipynb'
 features		  models	      reports
 layerwise_results.json  'nli-1(CPU).ipynb'   results


In [4]:
!ls /content/drive/MyDrive/NLI_HuBERT_Project/Dataset

ls: cannot access '/content/drive/MyDrive/NLI_HuBERT_Project/Dataset': No such file or directory


%cd /content/drive/MyDrive/NLI_HuBERT_Project
!mkdir -p data/raw
!unzip -q "Dataset/IndicAccentDB.zip" -d data/raw
!ls data/raw

In [5]:
!ls /content/drive/MyDrive/

 --71---97---103---97---110---32---82---101---100---100---121-gaganreddy001-gmail-com-2980360a-872.pdf
'Apex-(NLP-HuBert)'
 Assignment-1.zip
'CLV Documentation (1).gdoc'
'Colab Notebooks'
'Copy of Another copy of jupyter.ipynb'
 GaganReddy-HITAM-2024-BATCH-certificate.pdf
'Innofiesta fee payment  (1).pdf'
'Innofiesta fee payment .pdf'
 NLI_HuBERT_Proj-2025
 NLI_HuBERT_Project
 NLI_HuBERT_Project-20251111T063720Z-1-002
 NOTEBOOK_TO_SCRIPTS.md
'Offer letter .pdf'
 PROJECT_OVERVIEW.md
 PROJECT_REPORT.md
 SUBMISSION_READY.md


In [6]:
import os
import pandas as pd

root = "/content/drive/MyDrive/NLI_HuBERT_Project/data/raw"
rows = []

for lang_folder in os.listdir(root):
    lang_path = os.path.join(root, lang_folder)
    if os.path.isdir(lang_path):
        for root_dir, _, files in os.walk(lang_path):
            for f in files:
                if f.endswith(".wav"):
                    path = os.path.join(root_dir, f)
                    speaker_id = os.path.basename(root_dir)
                    label = lang_folder
                    rows.append([path, speaker_id, label])

df = pd.DataFrame(rows, columns=["wav_path", "speaker_id", "label"])
df.to_csv("metadata.csv", index=False)
print(f"✅ metadata.csv created with {len(df)} entries")
df.head(10)

✅ metadata.csv created with 8116 entries


,wav_path,speaker_id,label
0,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
1,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
2,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
3,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
4,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
5,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
6,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
7,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
8,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh
9,/content/drive/MyDrive/NLI_HuBERT_Project/data...,andhra_pradesh,andhra_pradesh


In [7]:
# Create speaker-disjoint manifests (run in Colab)
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from pathlib import Path
import os

# load metadata
meta_path = "metadata.csv"
df = pd.read_csv(meta_path)
print("Total rows in metadata:", len(df))

# ensure columns
if 'utt_id' not in df.columns:
    df['utt_id'] = df['wav_path'].apply(lambda p: Path(p).stem)

# normalize labels (optional)
df['label'] = df['label'].astype(str).str.strip().str.lower()  # lowercase for consistency

# map label -> idx
labels = sorted(df['label'].unique())
label2idx = {l:i for i,l in enumerate(labels)}
df['label_idx'] = df['label'].map(label2idx)

print("Labels found:", labels)
print("Counts per label:")
print(df['label'].value_counts())

# speaker-disjoint split params
TEST_SIZE = 0.10   # 10% test
VAL_SIZE = 0.10    # 10% val

# first split out test (by speaker)
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['speaker_id']))
df_test = df.iloc[test_idx].copy()
df_rem = df.iloc[train_idx].copy()

# split remaining into train/val
# compute val fraction relative to remaining
val_frac_of_rem = VAL_SIZE / (1.0 - TEST_SIZE)
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_rem, random_state=43)
train_idx2, val_idx = next(gss2.split(df_rem, groups=df_rem['speaker_id']))
df_val = df_rem.iloc[val_idx].copy()
df_train = df_rem.iloc[train_idx2].copy()

# add split column
df_train['split'] = 'train'
df_val['split']   = 'val'
df_test['split']  = 'test'

# combine and save
outdir = Path("data/manifests")
outdir.mkdir(parents=True, exist_ok=True)
df_all = pd.concat([df_train, df_val, df_test])
df_all.to_csv(outdir/"manifests.csv", index=False)
df_train.to_csv(outdir/"train.csv", index=False)
df_val.to_csv(outdir/"val.csv", index=False)
df_test.to_csv(outdir/"test.csv", index=False)

print("\nSaved manifests to", outdir)
print("Train / Val / Test sizes:", len(df_train), len(df_val), len(df_test))

# quick speaker-disjoint checks
train_spks = set(df_train['speaker_id'].unique())
val_spks   = set(df_val['speaker_id'].unique())
test_spks  = set(df_test['speaker_id'].unique())
print("Unique speakers (train/val/test):", len(train_spks), len(val_spks), len(test_spks))
print("train∩val:", len(train_spks & val_spks), "train∩test:", len(train_spks & test_spks), "val∩test:", len(val_spks & test_spks))

# per-label distribution in each split
print("\nPer-label counts in each split:")
print("TRAIN\n", df_train['label'].value_counts())
print("VAL\n", df_val['label'].value_counts())
print("TEST\n", df_test['label'].value_counts())

# if you want to inspect first few rows:
display(df_all.head(10))

Total rows in metadata: 8116
Labels found: ['andhra_pradesh', 'gujrat', 'jharkhand', 'karnataka', 'kerala', 'tamil']
Counts per label:
label
tamil             1840
andhra_pradesh    1794
karnataka         1686
kerala            1671
jharkhand          827
gujrat             298
Name: count, dtype: int64

Saved manifests to data/manifests
Train / Val / Test sizes: 4651 1671 1794
Unique speakers (train/val/test): 4 1 1
train∩val: 0 train∩test: 0 val∩test: 0

Per-label counts in each split:
TRAIN
 label
tamil        1840
karnataka    1686
jharkhand     827
gujrat        298
Name: count, dtype: int64
VAL
 label
kerala    1671
Name: count, dtype: int64
TEST
 label
andhra_pradesh    1794
Name: count, dtype: int64


,wav_path,speaker_id,label,utt_id,label_idx,split
1794,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_10_7,1,train
1795,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_1_12,1,train
1796,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_10_12,1,train
1797,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_1_11,1,train
1798,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_10_13,1,train
1799,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_1_2,1,train
1800,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_10_4,1,train
1801,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_10_10,1,train
1802,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_10_14,1,train
1803,/content/drive/MyDrive/NLI_HuBERT_Project/data...,gujrat,gujrat,Gujrat_speaker_01_1_6,1,train


In [ ]:
import os, librosa, numpy as np, pandas as pd
from pathlib import Path
from tqdm import tqdm

manifest = "data/manifests/manifests.csv"
out_dir = Path("features/mfcc")
out_dir.mkdir(parents=True, exist_ok=True)

def extract_mfcc(wav_path, sr=16000, n_mfcc=40, hop_length=160, n_fft=400):
    y, _ = librosa.load(wav_path, sr=sr)
    y = librosa.effects.preemphasis(y)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat = np.vstack([mfcc, delta, delta2])  # shape (n_feat, T)
    return feat.astype(np.float32)

df = pd.read_csv(manifest)
print(f"Extracting MFCCs for {len(df)} utterances...")

for i, row in tqdm(df.iterrows(), total=len(df)):
    utt_id = row["utt_id"]
    wav_path = row["wav_path"]
    save_path = out_dir / f"{utt_id}.npy"
    try:
        feat = extract_mfcc(wav_path)
        np.save(save_path, feat)
    except Exception as e:
        print(f"⚠️ Error {wav_path}: {e}")

print("✅ MFCC extraction complete!")
print("Sample file:", os.listdir(out_dir)[:5])

Extracting MFCCs for 8116 utterances...


100%|██████████| 8116/8116 [41:05<00:00,  3.29it/s]

✅ MFCC extraction complete!
Sample file: ['Tamil_speaker (1695).npy', 'Tamil_speaker (306).npy', 'Kerala_speaker_05_List50_Splitted_2.npy', 'Kerala_speaker_04_List43_Splitted_10.npy', 'Tamil_speaker (520).npy']


In [ ]:
import numpy as np
x = np.load('features/mfcc/' + os.listdir('features/mfcc')[0])
print(x.shape)

(120, 461)


In [ ]:
 %cd /content/drive/MyDrive/NLI_HuBERT_Project
!pwd
!ls

/content/drive/MyDrive/NLI_HuBERT_Project
/content/drive/MyDrive/NLI_HuBERT_Project
 data			  metadata.csv	     'nli-2(GPU).ipynb'
 features		  models	      reports
 layerwise_results.json  'nli-1(CPU).ipynb'   results


In [ ]:
!ls -lh features

total 12K
drwx------ 2 root root 4.0K Nov  9 05:23 hubert
drwx------ 2 root root 4.0K Nov 10 15:36 hubert_npy
drwx------ 2 root root 4.0K Nov  9 05:12 mfcc


In [ ]:
# Complete Deep Learning Training Script
# This includes CNN, BiLSTM, and Transformer architectures

# Run the training script
!python scripts/train_dl_models.py

python3: can't open file '/content/drive/MyDrive/NLI_HuBERT_Project/scripts/train_dl_models.py': [Errno 2] No such file or directory


In [ ]:
# Display the comparison plot
from IPython.display import Image, display
display(Image('results/dl_models_comparison.png'))

FileNotFoundError: No such file or directory: 'results/dl_models_comparison.png'

FileNotFoundError: No such file or directory: 'results/dl_models_comparison.png'

<IPython.core.display.Image object>

In [ ]:
# Display comprehensive experiments visualization
from IPython.display import Image, display
display(Image('results/comprehensive_experiments.png'))

FileNotFoundError: No such file or directory: 'results/comprehensive_experiments.png'

FileNotFoundError: No such file or directory: 'results/comprehensive_experiments.png'

<IPython.core.display.Image object>

In [ ]:
# Complete results comparison across all approaches
import json
import pandas as pd

print("="*90)
print(" "*25 + "COMPLETE PROJECT RESULTS")
print("="*90)

# Load all results
try:
    with open('models/mfcc_baseline_info.json', 'r') as f:
        mfcc_info = json.load(f)
    mfcc_acc = mfcc_info['accuracy'] * 100
except:
    mfcc_acc = "N/A"

try:
    with open('models/hubert_training_info.json', 'r') as f:
        hubert_rf_info = json.load(f)
    hubert_rf_acc = hubert_rf_info['accuracy'] * 100
except:
    hubert_rf_acc = "N/A"

try:
    with open('results/hubert_layerwise_results.json', 'r') as f:
        layer_results = json.load(f)
    best_layer_acc = max([r['accuracy'] for r in layer_results['layer_results']]) * 100
except:
    best_layer_acc = "N/A"

try:
    with open('results/dl_models_results.json', 'r') as f:
        dl_results = json.load(f)
    cnn_acc = dl_results['models']['CNN']['best_val_acc']
    bilstm_acc = dl_results['models']['BiLSTM']['best_val_acc']
    transformer_acc = dl_results['models']['Transformer']['best_val_acc']
except:
    cnn_acc = bilstm_acc = transformer_acc = "N/A"

# Create comparison table
results_data = [
    ["MFCC + Random Forest (Baseline)", mfcc_acc],
    ["HuBERT Layer 3 + Random Forest", hubert_rf_acc],
    ["HuBERT Best Layer + RF (Layer-wise)", best_layer_acc],
    ["HuBERT + CNN", cnn_acc],
    ["HuBERT + BiLSTM", bilstm_acc],
    ["HuBERT + Transformer", transformer_acc]
]

print(f"\n{'Approach':<45} {'Validation Accuracy':<20}")
print("-"*65)
for approach, acc in results_data:
    if isinstance(acc, (int, float)):
        print(f"{approach:<45} {acc:>18.2f}%")
    else:
        print(f"{approach:<45} {acc:>20}")

# Find best model
numeric_results = [(a, v) for a, v in results_data if isinstance(v, (int, float))]
if numeric_results:
    best_approach, best_acc = max(numeric_results, key=lambda x: x[1])
    print("\n" + "="*65)
    print(f"🏆 Best Model: {best_approach}")
    print(f"🏆 Best Accuracy: {best_acc:.2f}%")
    print("="*65)

print("\n📁 Generated Files:")
print("  • models/ - All trained models (.pth, .joblib)")
print("  • results/ - JSON results and visualizations")
print("  • features/ - Cached features for fast reloading")
print("\n✅ Project Complete!")
print("="*90)

---
## 7. Final Results Summary

All models trained and evaluated. Here's the complete comparison:

In [ ]:
# View experiment results
import json

with open('results/comprehensive_experiments.json', 'r') as f:
    exp_results = json.load(f)

print("="*80)
print("COMPREHENSIVE EXPERIMENTS - RESULTS SUMMARY")
print("="*80)

# Experiment 1
print("\n📊 EXPERIMENT 1: HuBERT vs MFCC (Random Forest)")
print("-"*80)
exp1 = exp_results['experiment_1']
print(f"Winner: {exp1['winner']}")
print(f"\nHuBERT Layer {exp1['hubert_layer']}:")
print(f"  Accuracy: {exp1['hubert_accuracy']:.2f}%")
print(f"  F1-Macro: {exp1['hubert_f1_macro']:.4f}")
print(f"\nMFCC:")
print(f"  Accuracy: {exp1['mfcc_accuracy']:.2f}%")
print(f"  F1-Macro: {exp1['mfcc_f1_macro']:.4f}")

# Experiment 2
print("\n\n📊 EXPERIMENT 2: Classifier Comparison (MFCC)")
print("-"*80)
exp2 = exp_results['experiment_2']
print(f"Best Classifier: {exp2['best_classifier']}")
print(f"Best Accuracy: {exp2['best_accuracy']:.2f}%\n")
print(f"{'Classifier':<25} {'Accuracy':<12} {'F1-Macro':<12}")
print("-"*50)
for clf_name, metrics in exp2['classifiers'].items():
    print(f"{clf_name:<25} {metrics['accuracy']:>10.2f}% {metrics['f1_macro']:>11.4f}")

# Experiment 3
print("\n\n📊 EXPERIMENT 3: Layer-wise Analysis with", exp2['best_classifier'])
print("-"*80)
exp3 = exp_results['experiment_3']
print(f"Best Layer: {exp3['best_layer']}")
print(f"Best Accuracy: {exp3['best_accuracy']:.2f}%")
print(f"MFCC Baseline: {exp3['mfcc_baseline_accuracy']:.2f}%")
print(f"\nLayer-wise accuracies:")
for layer, acc in enumerate(exp3['layer_accuracies']):
    marker = " ⭐ BEST" if layer == exp3['best_layer'] else ""
    print(f"  Layer {layer:2d}: {acc:>6.2f}%{marker}")

print("\n" + "="*80)

In [ ]:
# Run comprehensive experiments
# Note: First run will take time to load and cache 4,264 MFCC files
# Subsequent runs will be much faster due to caching

!python scripts/run_experiments.py

### Experimental Design

**Experiment 1: Feature Type Comparison**
- Compare HuBERT (Layer 3) vs MFCC using Random Forest
- Determine which feature representation is better

**Experiment 2: Classifier Comparison on MFCC**
- Test 8 different classifiers on MFCC features:
  - Random Forest, Gradient Boosting, Logistic Regression
  - SVM (Linear), SVM (RBF), K-Nearest Neighbors
  - Decision Tree, Naive Bayes
- Find the best classifier

**Experiment 3: Layer-wise Analysis with Best Classifier**
- Use winning classifier from Experiment 2
- Test on all 13 HuBERT layers (0-12)
- Compare with MFCC baseline
- Find optimal layer

---
## 6. Comprehensive Experiments & Comparison

Three key experiments comparing MFCC vs HuBERT features across multiple classifiers and layers.

In [ ]:
# View the complete training results
import json

with open('results/dl_models_results.json', 'r') as f:
    dl_results = json.load(f)

print("="*70)
print("DEEP LEARNING MODELS - FINAL RESULTS")
print("="*70)
print(f"\nDataset: {dl_results['num_samples']} samples, {dl_results['num_classes']} classes")
print(f"Training: {dl_results['epochs']} epochs, batch_size={dl_results['batch_size']}")
print(f"Device: {dl_results['device']}")
print(f"\nClasses: {', '.join(dl_results['classes'])}")
print("\n" + "-"*70)
print(f"{'Model':<20} {'Best Val Acc':<20} {'Final Acc':<20}")
print("-"*70)
for model_name, metrics in dl_results['models'].items():
    print(f"{model_name:<20} {metrics['best_val_acc']:>18.2f}% {metrics['final_acc']:>18.2f}%")
print("="*70)

### 5.1 Deep Learning Model Architectures

The training script implements three architectures:

**1. CNN (Convolutional Neural Network)**
- 3 Conv1D layers (64 → 128 → 256 filters)
- Batch normalization + MaxPooling
- 3 FC layers (512 → 256 → num_classes)
- Dropout: 0.5

**2. BiLSTM (Bidirectional LSTM)**
- 2-layer bidirectional LSTM (256 hidden units)
- Chunks 768-dim features into 48 sequences of 16-dim
- 3 FC layers (256 → 128 → num_classes)
- Dropout: 0.5

**3. Transformer Encoder**
- 4 layers, 8 attention heads
- d_model: 256, feedforward: 1024
- Positional encoding
- Global average pooling
- Dropout: 0.3

**Training Configuration:**
- Optimizer: Adam (lr=0.001)
- Loss: CrossEntropyLoss
- Batch size: 32
- Epochs: 50
- Device: CPU (auto-detects CUDA if available)

---
## 5. Deep Learning Models (CNN, BiLSTM, Transformer)

Training neural network architectures on HuBERT features for comparison with RF baseline.

In [ ]:
import os, librosa, numpy as np, pandas as pd
from pathlib import Path
from tqdm import tqdm

manifest = "data/manifests/manifests.csv"
out_dir = Path("features/mfcc")
out_dir.mkdir(parents=True, exist_ok=True)

def extract_mfcc(wav_path, sr=16000, n_mfcc=40, hop_length=160, n_fft=400):
    y, _ = librosa.load(wav_path, sr=sr)
    y = librosa.effects.preemphasis(y)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    feat = np.vstack([mfcc, delta, delta2])
    return feat.astype(np.float32)

df = pd.read_csv(manifest)
print(f"Extracting MFCCs for {len(df)} utterances...")

for i, row in tqdm(df.iterrows(), total=len(df)):
    utt_id = row["utt_id"]
    wav_path = row["wav_path"]
    save_path = out_dir / f"{utt_id}.npy"
    if save_path.exists():
        continue  # skip if already processed
    try:
        feat = extract_mfcc(wav_path)
        np.save(save_path, feat)
    except Exception as e:
        print(f"⚠️ Error {wav_path}: {e}")

print("✅ MFCC extraction complete!")
!ls -1 features/mfcc | head -n 10

In [ ]:
!ls -1 features/mfcc | wc -l
!ls -1 features/hubert | wc -l

In [ ]:
# ✅ Train a baseline classifier on MFCC features (fixed for your metadata)
import numpy as np, pandas as pd, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Paths
mfcc_dir = "features/mfcc"
meta = pd.read_csv("metadata.csv")  # has columns: wav_path, speaker_id, label

# Build dataset
X, y, ids = [], [], []

for _, row in meta.iterrows():
    # Extract base filename without extension (e.g., Andhra_speaker(29))
    utt = os.path.splitext(os.path.basename(row["wav_path"]))[0]
    label = row["label"]
    p = os.path.join(mfcc_dir, f"{utt}.npy")

    if os.path.exists(p):
        mf = np.load(p)
        X.append(mf.mean(axis=1))   # mean-pooling across frames
        y.append(label)
        ids.append(utt)

X = np.array(X)
y = np.array(y)
print("✅ Loaded MFCCs:", X.shape, "Labels:", len(y))

# Encode + scale
le = LabelEncoder()
y_enc = le.fit_transform(y)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Train Random Forest
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Evaluate
print("\n🎯 Accuracy:", accuracy_score(y_test, y_pred))
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Save model and preprocessing
os.makedirs("models", exist_ok=True)
joblib.dump(clf, "models/rf_mfcc.joblib")
joblib.dump(le, "models/label_encoder.joblib")
joblib.dump(scaler, "models/scaler.joblib")

print("\n💾 Model and preprocessing saved to /content/drive/MyDrive/NLI_HuBERT_Project/models/")

In [ ]:
import pandas as pd
meta = pd.read_csv("metadata.csv")
print(meta.columns)
meta.head()

In [ ]:
import os
print("Current working directory:", os.getcwd())
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/NLI_HuBERT_Project
!ls features/mfcc | head

In [ ]:
import joblib, numpy as np

# Load trained artifacts
clf = joblib.load("models/rf_mfcc.joblib")
le = joblib.load("models/label_encoder.joblib")
scaler = joblib.load("models/scaler.joblib")

# Load one MFCC file and predict
x = np.load("features/mfcc/Andhra_speaker (29).npy").mean(axis=1).reshape(1, -1)
x_scaled = scaler.transform(x)
pred = clf.predict(x_scaled)

print("Predicted Accent:", le.inverse_transform(pred)[0])

In [ ]:
def predict_accent(filename):
    x = np.load(filename).mean(axis=1).reshape(1, -1)
    x_scaled = scaler.transform(x)
    pred = clf.predict(x_scaled)
    print(f"{filename} → Predicted Accent:", le.inverse_transform(pred)[0])

# Example usage
predict_accent("features/mfcc/Andhra_speaker (46).npy")
predict_accent("features/mfcc/Kerala_speaker_01_10_15.npy")

In [ ]:
!ls features/mfcc | grep -i "Kerala" | head -n 20

In [ ]:
import random, glob

files = glob.glob("features/mfcc/Kerala_speaker_*.npy")
sample = random.choice(files)
print("Testing file:", sample)
predict_accent(sample)

In [ ]:

!echo "HuBERT files saved so far:" && ls -1 features/hubert | wc -l
!echo "MFCC files:" && ls -1 features/mfcc | wc -l

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Paths
manifest = "data/manifests/manifests.csv"
hubert_dir = "features/hubert"

# Load metadata
df = pd.read_csv(manifest)

# Build dataset from existing .npz files
X, y = [], []
for _, row in df.iterrows():
    utt = row["utt_id"]
    label = row["label"]
    npz_path = os.path.join(hubert_dir, f"{utt}.npz")
    if os.path.exists(npz_path):
        data = np.load(npz_path)
        pooled = data["pooled"]
        feat = pooled.flatten()  # flatten all layers
        X.append(feat)
        y.append(label)

X, y = np.array(X), np.array(y)
print("✅ Loaded HuBERT:", X.shape, "Labels:", len(y))

# Encode + scale
le = LabelEncoder()
y_enc = le.fit_transform(y)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

# Train
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Evaluate
from sklearn.utils.multiclass import unique_labels

acc = accuracy_score(y_test, y_pred)
print(f"🎯 Accuracy: {acc:.4f}")

# Only include labels that appear in y_test
present_classes = unique_labels(y_test)
present_names = le.inverse_transform(present_classes)

print("\n📊 Classification Report:\n",
      classification_report(y_test, y_pred, target_names=present_names))

In [ ]:
print("All labels:", list(le.classes_))
print("Labels present in current subset:", list(present_names))

In [ ]:
import pandas as pd, os, numpy as np
manifest = "data/manifests/manifests.csv"
df = pd.read_csv(manifest)

# count how many .npz per label we currently have
hubert_dir = "features/hubert"
counts = {}
for _, row in df.iterrows():
    utt = row["utt_id"]
    lab = row["label"]
    if os.path.exists(os.path.join(hubert_dir, f"{utt}.npz")):
        counts[lab] = counts.get(lab, 0) + 1

print("HuBERT counts by label (partial):")
for k,v in counts.items():
    print(f"  {k}: {v}")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.inverse_transform(sorted(set(y_test))), zero_division=0))

In [ ]:
MIN_SAMPLES = 30   # choose threshold
valid_labels = [lab for lab,c in counts.items() if c >= MIN_SAMPLES]
print("Using labels:", valid_labels)

# build X,y only for valid_labels
X, y = [], []
for _, row in df.iterrows():
    utt, lab = row["utt_id"], row["label"]
    if lab not in valid_labels:
        continue
    p = os.path.join(hubert_dir, f"{utt}.npz")
    if os.path.exists(p):
        pooled = np.load(p)['pooled']
        X.append(pooled.flatten())
        y.append(lab)

X = np.array(X); y = np.array(y)
# then encode/scale/train as before

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
clf.fit(X_train, y_train)

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
from collections import Counter
from imblearn.over_sampling import SMOTE

# show class counts
cnt = Counter(y_train)
print("Train class counts:", cnt)
min_count = min(cnt.values())

if min_count <= 1:
    print("SMOTE cannot run: a class has 1 or 0 samples. Use RandomOverSampler or remove tiny classes.")
else:
    k = min(5, min_count - 1)
    print("Using k_neighbors =", k)
    sm = SMOTE(random_state=42, k_neighbors=k)
    X_res, y_res = sm.fit_resample(X_train, y_train)
    print("Resampled shape:", X_res.shape, Counter(y_res))

In [ ]:
# Train + evaluate after SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder

# If your y are encoded ints (not strings) and you have a LabelEncoder object 'le' saved earlier,
# you can reverse map for nicer labels in reports. If not, we'll show integer class labels.
try:
    class_names = le.inverse_transform(sorted(set(y_test)))
except Exception:
    class_names = [str(c) for c in sorted(set(y_test))]

# Train RF on the resampled set
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
clf.fit(X_res, y_res)

# Eval on original test set
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}\n")

# Classification report (zero_division=0 prevents warnings turning into errors)
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred, labels=sorted(set(y_test)))
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (after SMOTE)");
plt.show()

# Save model
import os
os.makedirs("models", exist_ok=True)
joblib.dump(clf, "models/rf_hubert_smote.joblib")
print("Saved model -> models/rf_hubert_smote.joblib")

In [ ]:
# Full robust pipeline: rebuild features -> scale -> PCA -> optional SMOTE -> train -> eval -> save
import os, joblib, numpy as np, pandas as pd, time
from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

#  CONFIG
MANIFEST = "data/manifests/manifests.csv"   # manifest with utt_id,wav_path,label
HUBERT_DIR = "features/hubert"              # folder with .npz HuBERT files
USE_SMOTE = False                           # set True to attempt SMOTE oversampling (safe checks applied)
SMOTE_MAX_K = 5                             # maximum k_neighbors attempt (will reduce if class too small)
PCA_COMPONENTS = 256                        # target PCA dims (auto-capped)
RANDOM_STATE = 42
TEST_SIZE = 0.20
MODEL_OUT_DIR = "models"
#

start_time = time.time()
print("Loading manifest:", MANIFEST)
df = pd.read_csv(MANIFEST)
print("Total manifest rows:", len(df))

# 1) Build X,y only from existing HuBERT .npz files (keeps them aligned)
X_list, y_list, utts = [], [], []
missing = 0
for _, row in df.iterrows():
    utt = row["utt_id"]
    label = row["label"]
    path = os.path.join(HUBERT_DIR, f"{utt}.npz")
    if os.path.exists(path):
        data = np.load(path)
        # prefer 'pooled' key, fallback to first array in archive
        if "pooled" in data:
            pooled = data["pooled"]
        elif "pooled-p" in data:
            pooled = data["pooled-p"]
        else:
            keys = list(data.files)
            pooled = data[keys[0]]
        feat = pooled.flatten()
        X_list.append(feat)
        y_list.append(label)
        utts.append(utt)
    else:
        missing += 1

print(f"Found {len(X_list)} HuBERT files, missing {missing} (out of {len(df)})")
if len(X_list) == 0:
    raise RuntimeError("No HuBERT .npz files found. Check HUBERT_DIR and manifest.")

# 2) Convert to arrays and inspect
X = np.vstack(X_list)
y = np.array(y_list)
print("X.shape, y.shape =", X.shape, y.shape)

# 3) Encode labels
le = LabelEncoder().fit(y)
y_enc = le.transform(y)
print("Classes (all):", list(le.classes_))
print("Counts (all):", Counter(y_enc))

# 4) Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5) PCA (cap components if feature dim smaller)
n_comp = min(PCA_COMPONENTS, X_scaled.shape[1])
if n_comp < PCA_COMPONENTS:
    print(f"Warning: feature dim {X_scaled.shape[1]} < requested PCA {PCA_COMPONENTS}, using {n_comp}")
pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print("X_pca shape:", X_pca.shape)

# 6) Train/test split (stratify only if every class has >=2 samples)
counts = Counter(y_enc)
min_count = min(counts.values())
if min_count < 2:
    print("WARNING: At least one class has <2 samples; will NOT stratify split.")
    stratify_arg = None
else:
    stratify_arg = y_enc

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y_enc, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify_arg
)
print("Train/Test sizes:", X_train.shape[0], X_test.shape[0])
print("Train counts:", Counter(y_train))
print("Test counts:", Counter(y_test))

# 7) Optional: SMOTE (safe checks)
if USE_SMOTE:
    try:
        from imblearn.over_sampling import SMOTE
        # compute k based on smallest class in train
        train_counts = Counter(y_train)
        min_train = min(train_counts.values())
        if min_train <= 1:
            print("SMOTE cannot run: a class in training has <=1 samples. Skipping SMOTE.")
        else:
            k = min(SMOTE_MAX_K, min_train - 1)
            print(f"Applying SMOTE with k_neighbors={k} (min_train={min_train})")
            sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=k)
            X_train, y_train = sm.fit_resample(X_train, y_train)
            print("After SMOTE, train counts:", Counter(y_train))
    except Exception as e:
        print("SMOTE import or fit failed—continuing without SMOTE. Error:", e)

# 8) Train RandomForest (balanced)
clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
print("Training RandomForest...")
clf.fit(X_train, y_train)

# 9) Evaluate — safe printing of classification_report for only classes present in the test set
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nOverall test accuracy: {acc:.4f}")

# determine which encoded labels appear in the test set
present_labels = np.unique(y_test)
present_names = le.inverse_transform(present_labels)
print("Classes present in test:", list(present_names))
print("Test counts:", Counter(y_test))

print("\nClassification report (only classes present in test):")
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

# confusion matrix for only present classes
cm = confusion_matrix(y_test, y_pred, labels=present_labels)
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=present_names, yticklabels=present_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (present classes only)")
plt.show()

# 10) Save artifacts
os.makedirs(MODEL_OUT_DIR, exist_ok=True)
joblib.dump(clf, os.path.join(MODEL_OUT_DIR, "rf_hubert_final.joblib"))
joblib.dump(scaler, os.path.join(MODEL_OUT_DIR, "scaler_hubert.joblib"))
joblib.dump(pca, os.path.join(MODEL_OUT_DIR, "pca_hubert.joblib"))
joblib.dump(le, os.path.join(MODEL_OUT_DIR, "label_encoder.joblib"))
print("Saved: classifier, scaler, pca, label encoder ->", MODEL_OUT_DIR)

elapsed = time.time() - start_time
print(f"Done. Elapsed: {elapsed/60:.2f} minutes")

In [ ]:
!find /content/drive/MyDrive/NLI_HuBERT_Project -type f -name "*.csv"

In [ ]:
import pandas as pd

meta = pd.read_csv("/content/drive/MyDrive/NLI_HuBERT_Project/data/manifests/manifests.csv")
print(meta.shape)
print(meta.columns)
meta.head()

In [ ]:
!find /content/drive/MyDrive/NLI_HuBERT_Project -type d -name "mfcc"
!find /content/drive/MyDrive/NLI_HuBERT_Project -type d -name "hubert"

In [ ]:
MFCC_DIR = "features/mfcc"
HUBERT_DIR = "features/hubert"

In [ ]:
!ls -1 /content/drive/MyDrive/NLI_HuBERT_Project/features/mfcc | wc -l
!ls -1 /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert | wc -l

In [ ]:
# Combined MFCC + HuBERT training & simple layer analysis
# Run in Colab. Assumes:
# - manifest: /content/drive/MyDrive/NLI_HuBERT_Project/data/manifests/manifests.csv
# - MFCC files: features/mfcc/*.npy
# - HuBERT files: features/hubert/*.npz
# - models saved to ./models/

import os, numpy as np, pandas as pd, joblib
from collections import Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

MANIFEST = "/content/drive/MyDrive/NLI_HuBERT_Project/data/manifests/manifests.csv"
MFCC_DIR = "features/mfcc"
HUBERT_DIR = "features/hubert"
os.makedirs("models", exist_ok=True)

meta = pd.read_csv(MANIFEST)
print("Manifest rows:", len(meta))
print(meta.columns)
print()

#  Utility: protection if labels missing
if "utt_id" not in meta.columns:
    raise RuntimeError("manifest missing 'utt_id' column")

# common label encoder
le = LabelEncoder()


# 1) Speaker-wise MFCC baseline

print("1) Building MFCC dataset (speaker-wise split)...")
X_mfcc, y_mfcc, groups = [], [], []
for _, r in meta.iterrows():
    utt = r["utt_id"]
    p = os.path.join(MFCC_DIR, f"{utt}.npy")
    if os.path.exists(p):
        arr = np.load(p)               # shape (n_feats, T)
        feat = arr.mean(axis=1)        # pooled vector (n_feats,)
        X_mfcc.append(feat)
        y_mfcc.append(r["label"])
        # speaker id: prefer speaker_id column if present
        groups.append(r["speaker_id"] if "speaker_id" in meta.columns else utt.split('(')[0].strip())
X_mfcc = np.array(X_mfcc); y_mfcc = np.array(y_mfcc); groups = np.array(groups)
print("MFCC samples:", X_mfcc.shape, "labels:", Counter(y_mfcc))

if len(X_mfcc) == 0:
    print("No MFCC files found. Skip MFCC training.")
else:
    # speaker-wise split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X_mfcc, y_mfcc, groups))
    Xtr, Xte = X_mfcc[train_idx], X_mfcc[test_idx]
    ytr, yte = y_mfcc[train_idx], y_mfcc[test_idx]

    # encode/scale/train
    le_mfcc = LabelEncoder().fit(ytr)
    ytr_enc = le_mfcc.transform(ytr)
    yte_enc = le_mfcc.transform(yte)
    scaler_mfcc = StandardScaler().fit(Xtr)
    Xtr_s = scaler_mfcc.transform(Xtr)
    Xte_s = scaler_mfcc.transform(Xte)

    clf_mfcc = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
    clf_mfcc.fit(Xtr_s, ytr_enc)
    ypred = clf_mfcc.predict(Xte_s)
    acc = accuracy_score(yte_enc, ypred)
    print(f"MFCC speaker-wise test accuracy: {acc:.4f}")
    print(classification_report(yte_enc, ypred, target_names=le_mfcc.classes_, zero_division=0))

    # confusion
    cm = confusion_matrix(yte_enc, ypred)
    plt.figure(figsize=(7,5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=le_mfcc.classes_, yticklabels=le_mfcc.classes_, cmap="Blues")
    plt.title("MFCC: Speaker-wise Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.show()

    # save
    joblib.dump(clf_mfcc, "models/rf_mfcc_speakerwise.joblib")
    joblib.dump(le_mfcc, "models/label_encoder_mfcc.joblib")
    joblib.dump(scaler_mfcc, "models/scaler_mfcc.joblib")
    print("Saved MFCC model + artifacts to models/")


# 2) HuBERT baseline (available .npz only)

print("\n2) Building HuBERT dataset from existing .npz files...")
X_h, y_h, utts = [], [], []
missing = 0
for _, r in meta.iterrows():
    utt = r["utt_id"]
    p = os.path.join(HUBERT_DIR, f"{utt}.npz")
    if os.path.exists(p):
        d = np.load(p)
        # common key from save: 'pooled' (layers x dim), otherwise take first array
        if "pooled" in d:
            pooled = d["pooled"]
        else:
            # fallback pick first array
            pooled = d[d.files[0]]
        # Use full flattened pooled (all layers concatenated) for baseline
        feat = pooled.flatten()
        X_h.append(feat)
        y_h.append(r["label"])
        utts.append(utt)
    else:
        missing += 1

X_h = np.vstack(X_h) if len(X_h)>0 else np.empty((0,0))
y_h = np.array(y_h)
print(f"HuBERT files found: {len(X_h)}   missing: {missing}")
if len(X_h) == 0:
    print("No HuBERT .npz files found. Skip HuBERT training.")
else:
    # encode labels (fit on huBERT subset)
    le_h = LabelEncoder().fit(y_h)
    y_h_enc = le_h.transform(y_h)

    # scale + PCA (reduce to <=256)
    scaler_h = StandardScaler().fit(X_h)
    Xh_s = scaler_h.transform(X_h)
    n_comp = min(256, Xh_s.shape[1])
    pca = PCA(n_components=n_comp, random_state=42).fit(Xh_s)
    Xh_pca = pca.transform(Xh_s)
    print("X_h shape, PCA dim:", X_h.shape, "->", Xh_pca.shape)

    # stratify only if all classes have >=2 samples
    from collections import Counter
    cc = Counter(y_h_enc)
    stratify = y_h_enc if min(cc.values()) >= 2 else None

    Xtr, Xte, ytr, yte = train_test_split(Xh_pca, y_h_enc, test_size=0.2, stratify=stratify, random_state=42)
    clf_h = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
    clf_h.fit(Xtr, ytr)
    ypred_h = clf_h.predict(Xte)
    acc_h = accuracy_score(yte, ypred_h)
    print(f"HuBERT (flatten-pooled -> PCA{n_comp}) test accuracy: {acc_h:.4f}")
    print(classification_report(yte, ypred_h, target_names=le_h.classes_, zero_division=0))

    # confusion
    cm = confusion_matrix(yte, ypred_h)
    plt.figure(figsize=(7,5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=le_h.classes_, yticklabels=le_h.classes_, cmap="Blues")
    plt.title("HuBERT (PCA) Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.show()

    # save
    joblib.dump(clf_h, "models/rf_hubert.joblib")
    joblib.dump(scaler_h, "models/scaler_hubert.joblib")
    joblib.dump(pca, "models/pca_hubert.joblib")
    joblib.dump(le_h, "models/label_encoder_hubert.joblib")
    print("Saved HuBERT model + artifacts to models/")

# 3) Quick layer-by-layer HuBERT check (which single layer is best?)

# This uses the 'pooled' array shaped (n_layers, hidden_dim) saved in each .npz.
# It will evaluate each layer as a feature vector (hidden_dim) and report test accuracy.
if len(X_h) == 0:
    print("\nNo HuBERT features to do layer analysis.")
else:
    print("\n3) Layer-by-layer HuBERT quick analysis (uses only utterances that had .npz files)")
    # rebuild feature lists per layer
    # first read one example to get num_layers
    sample = np.load(os.path.join(HUBERT_DIR, utts[0]))
    pooled0 = sample["pooled"] if "pooled" in sample else sample[sample.files[0]]
    n_layers = pooled0.shape[0]
    print("Number of pooled layers found:", n_layers)

    layer_scores = []
    for layer in range(n_layers):
        X_layer = []
        for utt in utts:
            d = np.load(os.path.join(HUBERT_DIR, utt))
            pooled = d["pooled"] if "pooled" in d else d[d.files[0]]
            vec = pooled[layer]            # shape (hidden_dim,)
            X_layer.append(vec)
        X_layer = np.array(X_layer)
        # scale + simple split
        sc = StandardScaler().fit(X_layer)
        Xs = sc.transform(X_layer)
        n_comp = min(128, Xs.shape[1])  # small PCA if dim large
        if n_comp < Xs.shape[1]:
            pc = PCA(n_components=n_comp, random_state=42).fit(Xs)
            Xs = pc.transform(Xs)
        # split
        strat = y_h_enc if min(Counter(y_h_enc).values())>=2 else None
        Xtr, Xte, ytr, yte = train_test_split(Xs, y_h_enc, test_size=0.2, stratify=strat, random_state=42)
        clf_tmp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
        clf_tmp.fit(Xtr, ytr)
        acc_tmp = accuracy_score(yte, clf_tmp.predict(Xte))
        layer_scores.append((layer, acc_tmp))
        print(f"layer {layer:02d} -> acc {acc_tmp:.4f}")

    layer_scores = sorted(layer_scores, key=lambda x: x[1], reverse=True)
    print("\nTop layers by accuracy (layer, acc):")
    for l, s in layer_scores[:6]:
        print(l, round(s,4))

    best_layer = layer_scores[0][0]
    print(f"\nBest single layer: {best_layer} (you could re-run using pooled[{best_layer}] for final model)")

In [ ]:
# list the project root and feature folders
!ls -la /content/drive/MyDrive | head -n 50
!ls -la "/content/drive/MyDrive/NLI_HuBERT_Project" || true
!ls -la "/content/drive/MyDrive/NLI_HuBERT_Project/features" || true
!ls -la "/content/drive/MyDrive/NLI_HuBERT_Project/features/mfcc" | head -n 20 || true
!ls -la "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert" | head -n 20 || true

In [ ]:
# Speaker-wise MFCC baseline (paste & run in Colab/Jupyter)
import os
import numpy as np
import pandas as pd
import joblib

from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# CONFIG - edit paths if needed
META = "metadata.csv"            # path to your manifest/metadata
MFCC_DIR = "features/mfcc"       # where utt.npy mfcc files are stored
RANDOM_STATE = 42
TEST_SIZE = 0.20

# Load metadata
if not os.path.exists(META):
    raise FileNotFoundError(f"metadata file not found: {META}")
meta = pd.read_csv(META)
print("Metadata rows:", len(meta))

# Columns expected: utt_id, speaker_id, label (change below if different)
utt_col = "utt_id"
spk_col = "speaker_id"
lab_col = "label"

# Build pooled MFCC features dataset (mean pooling across time)
X, y, groups, utts = [], [], [], []
missing = 0
for _, r in meta.iterrows():
    utt = str(r[utt_col])
    spk = str(r[spk_col])
    lab = r[lab_col]
    p = os.path.join(MFCC_DIR, f"{utt}.npy")
    if not os.path.exists(p):
        missing += 1
        continue
    arr = np.load(p)  # expected shape (n_mfcc, T)
    if arr.ndim == 1:
        feat = arr  # already pooled maybe
    else:
        feat = arr.mean(axis=1)  # per-file pooled vector
    X.append(feat)
    y.append(lab)
    groups.append(spk)
    utts.append(utt)

print(f"Missing MFCC files: {missing}")
X = np.array(X)
y = np.array(y)
groups = np.array(groups)
print("Loaded features:", X.shape, "Labels distribution:", Counter(y))

# Simple label encoding
le = LabelEncoder().fit(y)
y_enc = le.transform(y)

# Speaker-wise split
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y_enc, groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_enc[train_idx], y_enc[test_idx]
print("Train / Test sizes:", X_train.shape, X_test.shape)
print("Train speakers:", len(set(groups[train_idx])), "Test speakers:", len(set(groups[test_idx])))

# Scale
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# Train RF
clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
clf.fit(X_train_s, y_train)

# Eval
y_pred = clf.predict(X_test_s)
acc = accuracy_score(y_test, y_pred)
print(f"Speaker-wise test Accuracy: {acc:.4f}")
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(le.classes_)))
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("MFCC RF - Speaker-wise Confusion Matrix")
plt.tight_layout()
plt.show()

# Save model + scaler for quick reuse
os.makedirs("models", exist_ok=True)
joblib.dump({"clf": clf, "scaler": scaler, "label_encoder": le}, "models/mfcc_rf_speakerwise.joblib")
print("Saved models/mfcc_rf_speakerwise.joblib")

In [ ]:
# Robust layer-by-layer HuBERT evaluation (handles different manifest schemas)
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

MANIFEST = "metadata.csv"
HUBERT_DIR = "features/hubert"   # change if your .npz files are elsewhere
TEST_SIZE = 0.20
RANDOM_STATE = 42

# -- load manifest
if not os.path.exists(MANIFEST):
    raise FileNotFoundError(f"manifest not found at {MANIFEST}")
df = pd.read_csv(MANIFEST)
print("Manifest rows:", len(df))
print("Manifest columns:", df.columns.tolist())

# -- detect useful columns
candidate_utt_cols = ["utt_id", "utt", "id", "filename", "file", "wav", "wav_name", "wav_path", "path", "audio_path", "audio"]
candidate_speaker_cols = ["speaker_id", "speaker", "spk", "speakerID", "spk_id"]
candidate_label_cols = ["label", "lang", "language", "target"]

utt_col = next((c for c in candidate_utt_cols if c in df.columns), None)
spk_col = next((c for c in candidate_speaker_cols if c in df.columns), None)
lab_col = next((c for c in candidate_label_cols if c in df.columns), None)

print("Detected columns -> utt_col:", utt_col, "spk_col:", spk_col, "lab_col:", lab_col)
if lab_col is None:
    raise KeyError("Could not detect a label column in the manifest. Edit the manifest or update candidate_label_cols.")

# helper: get utt id (file-stem) from row
def get_utt_from_row(row):
    # If there's a wav_path or path column, use its file stem
    if utt_col in df.columns and utt_col and pd.notna(row.get(utt_col)):
        val = row.get(utt_col)
        # if this is a path, take the stem
        p = Path(str(val))
        if p.suffix:  # has extension -> it's a path
            return p.stem
        else:
            return str(val)
    # fallback: look for wav_path-like columns
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).stem
    # final fallback: use index
    return str(row.name)

def get_speaker_from_row(row):
    if spk_col and spk_col in df.columns and pd.notna(row.get(spk_col)):
        return str(row.get(spk_col))
    # try to infer from wav_path folder name if present
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            p = Path(str(row.get(c)))
            parent = p.parent.name
            return parent
    # fallback
    return "unknown"

#  build fast index of hubert .npz files
hubert_index = {}
hubdir = Path(HUBERT_DIR)
if not hubdir.exists():
    raise FileNotFoundError(f"HuBERT dir not found: {HUBERT_DIR}")
for p in hubdir.rglob("*.npz"):
    hubert_index[p.stem] = p
print("Indexed hubert .npz files:", len(hubert_index))

# -- collect pooled per utt into per-layer lists
layer_feats = defaultdict(list)
labels = []
groups = []
missing = 0
n_layers_found = None
collected_utts = []

for _, row in df.iterrows():
    utt = get_utt_from_row(row)
    spk = get_speaker_from_row(row)
    lab = row[lab_col]
    # try direct match in index
    p = hubert_index.get(utt)
    if p is None:
        # try common normalization variants
        p = hubert_index.get(utt.replace(" ", "_")) or hubert_index.get(utt.replace("_"," "))
    if p is None:
        missing += 1
        continue
    try:
        data = np.load(p)
    except Exception as e:
        print("Failed loading", p, ":", e); missing += 1; continue

    # Parse pooled representation robustly
    pooled = None
    if "pooled" in data:
        pooled = data["pooled"]
    elif "hidden_states" in data:
        hs = data["hidden_states"]
        # hidden_states may be list-like; make array of mean(token dim)
        pooled = np.array([h.mean(axis=0) for h in hs])
    else:
        # try to find a 2D array among arrays (layers x dim)
        arrays = [data[k] for k in data.files]
        # choose the first 2D array with shape (layers, dim)
        chosen = None
        for a in arrays:
            if getattr(a, "ndim", 0) == 2:
                chosen = a; break
        if chosen is not None:
            pooled = chosen
        else:
            # if only 1D arrays exist, treat as single-layer pooled
            if len(arrays) == 1 and arrays[0].ndim == 1:
                pooled = arrays[0][None, :]
    if pooled is None:
        # cannot parse this file
        print("Warning: unknown .npz layout for", p, "files:", data.files)
        missing += 1
        continue

    # record n_layers
    if n_layers_found is None:
        n_layers_found = pooled.shape[0]
        print("Detected n_layers:", n_layers_found)
    # only accept pooled arrays matching detected n_layers
    if pooled.ndim == 2 and pooled.shape[0] == n_layers_found:
        for l in range(pooled.shape[0]):
            layer_feats[l].append(pooled[l])
        labels.append(lab)
        groups.append(spk)
        collected_utts.append(utt)
    else:
        # skip mismatched shapes
        missing += 1

print("Collected utterances with pooled:", len(collected_utts), "missing/skipped:", missing)
if len(collected_utts) == 0:
    raise RuntimeError("No usable HuBERT pooled files found. Check HUBERT_DIR and manifest utt naming.")

#  prepare arrays aligned
labels = np.array(labels)
groups = np.array(groups)
print("Label counts (collected):", Counter(labels))

#  encode labels
le = LabelEncoder().fit(labels)
labels_enc = le.transform(labels)

#  evaluate per-layer with speaker-wise split
results = {}
for l in sorted(layer_feats.keys()):
    X = np.vstack(layer_feats[l])  # shape (N, dim)
    # speaker-wise split using collected groups
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr, te = next(gss.split(X, labels_enc, groups))
    X_tr, X_te = X[tr], X[te]
    y_tr, y_te = labels_enc[tr], labels_enc[te]
    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr); X_te_s = scaler.transform(X_te)
    clf = LogisticRegression(max_iter=2000, multi_class='multinomial', solver='saga')
    clf.fit(X_tr_s, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te_s))
    results[l] = acc
    print(f"Layer {l:02d} -> acc: {acc:.4f}")

# plot
layers, accs = zip(*sorted(results.items()))
plt.figure(figsize=(9,4))
plt.plot(layers, accs, marker='o')
plt.xlabel("HuBERT layer")
plt.ylabel("Speaker-wise test accuracy")
plt.title("Layer-wise HuBERT accuracy (logreg, pooled vectors)")
plt.grid(True)
plt.show()

best_layer = max(results, key=results.get)
print("Best layer:", best_layer, "acc:", results[best_layer])

In [ ]:
import pandas as pd, os, numpy as np
df = pd.read_csv("metadata.csv")

hubert_dir = "features/hubert"
matched = 0
for _, row in df.iterrows():
    stem = os.path.splitext(os.path.basename(row['wav_path']))[0]
    if os.path.exists(os.path.join(hubert_dir, f"{stem}.npz")):
        matched += 1
print("Matched .npz files:", matched, "of", len(df))

In [ ]:
import os

hubert_dir = "features/hubert"
files = [f for f in os.listdir(hubert_dir) if f.endswith(".npz")]
print("Found", len(files), "files. Example few:")
print(files[:10])  # shows the first 10 filenames

In [ ]:
import numpy as np
sample = np.load("features/hubert/Andhra_speaker (848).npz")
print("Keys:", sample.files)
for k in sample.files:
    arr = sample[k]
    print(k, arr.shape, arr.dtype, "mean:", arr.mean(), "std:", arr.std())

In [ ]:
# Train/evaluate RF on best HuBERT layer (speaker-wise)
import os, json
from pathlib import Path
import numpy as np, pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# CONFIG - edit if needed
MANIFEST = "metadata.csv"
HUBERT_DIR = "features/hubert"
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
TEST_SIZE = 0.20
RANDOM_STATE = 42

# Load manifest & detect columns (robust)
df = pd.read_csv(MANIFEST)
candidate_utt_cols = ["utt_id","utt","id","filename","file","wav","wav_name","wav_path","path","audio_path","audio"]
candidate_speaker_cols = ["speaker_id","speaker","spk","speakerID","spk_id"]
candidate_label_cols = ["label","lang","language","target"]

utt_col = next((c for c in candidate_utt_cols if c in df.columns), None)
spk_col = next((c for c in candidate_speaker_cols if c in df.columns), None)
lab_col = next((c for c in candidate_label_cols if c in df.columns), None)
if lab_col is None:
    raise KeyError("No label column detected in manifest. Edit manifest or update candidate_label_cols.")

def get_utt_from_row(row):
    if utt_col and utt_col in df.columns and pd.notna(row.get(utt_col)):
        val = row.get(utt_col)
        p = Path(str(val))
        return p.stem if p.suffix else str(val)
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).stem
    return str(row.name)

def get_spk_from_row(row):
    if spk_col and spk_col in df.columns and pd.notna(row.get(spk_col)):
        return str(row.get(spk_col))
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).parent.name
    return "unknown"

# Build hubert index
hubdir = Path(HUBERT_DIR)
if not hubdir.exists():
    raise FileNotFoundError(f"HuBERT features directory not found: {HUBERT_DIR}")
hubert_index = {p.stem: p for p in hubdir.rglob("*.npz")}
print("Indexed .npz files:", len(hubert_index))

# Try to load previous layer results file (optional)
best_layer = None
if os.path.exists("layerwise_results.json"):
    with open("layerwise_results.json","r") as f:
        res = json.load(f)
    # res should be dict {layer_idx: accuracy}
    if isinstance(res, dict) and len(res)>0:
        # keys may be strings; convert to int
        res_int = {int(k): float(v) for k,v in res.items()}
        best_layer = max(res_int, key=res_int.get)
        print("Best layer from layerwise_results.json:", best_layer, "acc:", res_int[best_layer])

# If not found, compute a light per-layer accuracy to pick best (only if needed)
if best_layer is None:
    print("No saved layer results found — doing quick per-layer scan to pick best layer (speaker-wise).")
    # collect pooled arrays for matching utts
    utts = []
    utt_to_pooled = {}
    n_layers_detected = None
    for _, row in df.iterrows():
        utt = get_utt_from_row(row)
        p = hubert_index.get(utt) or hubert_index.get(utt.replace(" ", "_"))
        if not p:
            continue
        try:
            data = np.load(p)
        except:
            continue
        pooled = None
        if "pooled" in data:
            pooled = data["pooled"]
        elif "hidden_states" in data:
            pooled = np.array([h.mean(axis=0) for h in data["hidden_states"]])
        else:
            arrays = [data[k] for k in data.files]
            chosen = next((a for a in arrays if getattr(a,'ndim',0)==2), None)
            if chosen is not None:
                pooled = chosen
            elif len(arrays)==1 and arrays[0].ndim==1:
                pooled = arrays[0][None,:]
        if pooled is None:
            continue
        if n_layers_detected is None:
            n_layers_detected = pooled.shape[0]
        if pooled.ndim==2 and pooled.shape[0]==n_layers_detected:
            utt_to_pooled[utt] = pooled
            utts.append(utt)
    if len(utts)==0:
        raise RuntimeError("No usable pooled HuBERT files found.")
    print("Collected pooled for utts:", len(utts), "layers detected:", n_layers_detected)
    # per-layer quick eval (logreg)
    from sklearn.linear_model import LogisticRegression
    layer_scores = {}
    labels_all = []
    groups_all = []
    for utt in utts:
        row = df[df.apply(lambda r: get_utt_from_row(r)==utt, axis=1)]
        if row.shape[0]==0:
            labels_all.append(None); groups_all.append("unknown")
        else:
            labels_all.append(str(row.iloc[0][lab_col]))
            groups_all.append(get_spk_from_row(row.iloc[0]))
    labels_arr = np.array(labels_all)
    groups_arr = np.array(groups_all)
    # encode labels
    le_tmp = LabelEncoder().fit(labels_arr)
    y_enc = le_tmp.transform(labels_arr)
    for l in range(n_layers_detected):
        Xl = np.vstack([utt_to_pooled[u][l] for u in utts])
        # skip if dimension too large vs samples
        scalerL = StandardScaler().fit(Xl)
        Xl_s = scalerL.transform(Xl)
        # PCA small
        pcaL = PCA(n_components=min(64, Xl_s.shape[1]), random_state=42).fit(Xl_s)
        Xl_p = pcaL.transform(Xl_s)
        # speaker-wise split
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        try:
            tr, te = next(gss.split(Xl_p, y_enc, groups_arr))
        except Exception as e:
            layer_scores[l] = 0.0
            continue
        clf = LogisticRegression(max_iter=2000, multi_class='multinomial', solver='saga')
        clf.fit(Xl_p[tr], y_enc[tr])
        acc = accuracy_score(y_enc[te], clf.predict(Xl_p[te]))
        layer_scores[l] = acc
        print(f"Layer {l:02d} quick-acc: {acc:.4f}")
    best_layer = max(layer_scores, key=layer_scores.get)
    print("Picked best layer from quick scan:", best_layer, "acc:", layer_scores[best_layer])
    # save quick results for reproducibility
    with open("layerwise_results.json","w") as f:
        json.dump(layer_scores, f)

# Now build dataset using the best_layer pooled vector (one vector per utt)
X_list = []; y_list = []; groups = []; utt_list = []
for _, row in df.iterrows():
    utt = get_utt_from_row(row)
    p = hubert_index.get(utt) or hubert_index.get(utt.replace(" ", "_"))
    if not p:
        continue
    data = np.load(p)
    pooled = None
    if "pooled" in data:
        pooled = data["pooled"]
    elif "hidden_states" in data:
        pooled = np.array([h.mean(axis=0) for h in data["hidden_states"]])
    else:
        arrays = [data[k] for k in data.files]
        chosen = next((a for a in arrays if getattr(a,'ndim',0)==2), None)
        if chosen is not None:
            pooled = chosen
        elif len(arrays)==1 and arrays[0].ndim==1:
            pooled = arrays[0][None,:]
    if pooled is None:
        continue
    if pooled.ndim==2 and pooled.shape[0] > best_layer:
        vec = pooled[best_layer]   # (dim,)
    else:
        # fallback: mean across layers then use that
        vec = pooled.mean(axis=0) if pooled.ndim==2 else pooled
    X_list.append(vec)
    y_list.append(str(row[lab_col]))
    groups.append(get_spk_from_row(row))
    utt_list.append(utt)

X = np.vstack(X_list)
y = np.array(y_list)
groups = np.array(groups)
print("Built dataset:", X.shape, "labels:", Counter(y))

# encode labels and speaker-wise split
le = LabelEncoder().fit(y)
y_enc = le.transform(y)
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
tr, te = next(gss.split(X, y_enc, groups))
Xtr, Xte = X[tr], X[te]; ytr, yte = y_enc[tr], y_enc[te]

# scale + PCA (optional) + RF
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
pca = PCA(n_components=min(256, Xtr_s.shape[1]), random_state=42).fit(Xtr_s)
Xtr_p, Xte_p = pca.transform(Xtr_s), pca.transform(Xte_s)

clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
clf.fit(Xtr_p, ytr)
ypred = clf.predict(Xte_p)

acc = accuracy_score(yte, ypred)
print(f"HuBERT best-layer ({best_layer}) speaker-wise accuracy: {acc:.4f}")
print("Classification report:")
print(classification_report(yte, ypred, target_names=le.classes_, zero_division=0))

# confusion matrix
cm = confusion_matrix(yte, ypred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"HuBERT layer {best_layer} - Confusion Matrix")
plt.show()

# Save model artifacts
joblib.dump({"clf": clf, "scaler": scaler, "pca": pca, "label_encoder": le, "best_layer": best_layer}, MODEL_DIR / "hubert_bestlayer_rf.joblib")
print("Saved model to", MODEL_DIR / "hubert_bestlayer_rf.joblib")

In [ ]:
import pandas as pd, numpy as np, os
from collections import Counter, defaultdict
df = pd.read_csv("metadata.csv")

# detect utt & spk columns used earlier (fast)
def get_utt(r):
    for c in ["utt_id","wav_path","path","audio_path","filename","file","wav"]:
        if c in df.columns and pd.notna(r.get(c)):
            return str(r[c]).split("/")[-1].split(".")[0]
    return str(r.name)
def get_spk(r):
    for c in ["speaker_id","speaker","spk"]:
        if c in df.columns and pd.notna(r.get(c)):
            return str(r[c])
    # try parent folder from wav_path
    if "wav_path" in df.columns and pd.notna(r.get("wav_path")):
        return str(r["wav_path"]).split("/")[-2]
    return "unknown"

# build counts for only those utts for which .npz exists
hubert_dir = "features/hubert"
present = []
for _, r in df.iterrows():
    utt = get_utt(r)
    if os.path.exists(os.path.join(hubert_dir, f"{utt}.npz")) or os.path.exists(os.path.join(hubert_dir, f"{utt.replace(' ','_')}.npz")):
        present.append((utt, get_spk(r), r.get("label")))

print("Total manifest rows:", len(df))
print("Matched .npz for utts:", len(present))

# counts per label and per speaker-per-label
label_counts = Counter([t[2] for t in present])
spk_per_label = defaultdict(set)
for utt, spk, lab in present:
    spk_per_label[lab].add(spk)

print("Label counts (top):")
for k,v in label_counts.most_common():
    print(f"  {k}: {v} utts, {len(spk_per_label[k])} speakers")

In [ ]:
# safer training snippet
MIN_UTTS = 6   # remove classes with fewer than this many utts (tuneable)
# Build full dataset from best_layer as you did (X_list, y_list, groups)
# (Assume X_list, y_list, groups were built earlier as in your script.)
import numpy as np
from collections import Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Filter rare classes
counts = Counter(y_list)
keep_labels = {lab for lab,c in counts.items() if c >= MIN_UTTS}
print("Original class counts:", counts)
print("Keeping labels (>= {} utts): {}".format(MIN_UTTS, sorted(keep_labels)))

Xf, yf, gf = [], [], []
for x,y,g in zip(X_list, y_list, groups):
    if y in keep_labels:
        Xf.append(x); yf.append(y); gf.append(g)
Xf = np.vstack(Xf); yf = np.array(yf); gf = np.array(gf)
print("After filtering dataset shape:", Xf.shape, "labels:", Counter(yf))

# If dataset too small after filtering, lower MIN_UTTS or group small classes.
if Xf.shape[0] < 20:
    raise RuntimeError("Too few samples after filtering. Lower MIN_UTTS or add more data.")

# Encode & split speaker-wise
le = LabelEncoder().fit(yf)
y_enc = le.transform(yf)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, te = next(gss.split(Xf, y_enc, gf))
Xtr, Xte = Xf[tr], Xf[te]; ytr, yte = y_enc[tr], y_enc[te]
print("Train/Test sizes:", Xtr.shape, Xte.shape, "train labels:", Counter(ytr), "test labels:", Counter(yte))

# Scale
scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr); Xte_s = scaler.transform(Xte)

# PCA: choose n_components safely
max_components = min(256, Xtr_s.shape[1], Xtr_s.shape[0]-1)
if max_components <= 0:
    print("Skipping PCA because not enough train samples.")
    Xtr_p, Xte_p = Xtr_s, Xte_s
else:
    pca = PCA(n_components=max_components, random_state=42).fit(Xtr_s)
    Xtr_p, Xte_p = pca.transform(Xtr_s), pca.transform(Xte_s)
    print("PCA applied, components:", max_components)

# Train RF
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
clf.fit(Xtr_p, ytr)
yp = clf.predict(Xte_p)
print("Acc:", accuracy_score(yte, yp))
print(classification_report(yte, yp, target_names=le.classes_, zero_division=0))

In [ ]:
# Robust end-to-end: layer scan -> pick best HuBERT layer -> train speaker-wise RF -> save artifacts
import os, json
from pathlib import Path
import numpy as np, pandas as pd
from collections import Counter, defaultdict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

#  CONFIG
MANIFEST = "metadata.csv"             # path to manifest (change if needed)
HUBERT_DIR = "features/hubert"        # folder containing .npz HuBERT files
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
TEST_SIZE = 0.20
RANDOM_STATE = 42
MIN_UTTS = 6        # remove classes with fewer than this many utts (tuneable)
MAX_SPLIT_TRIES = 500
MIN_PER_CLASS_IN_TEST = 1   # require at least this many examples per class in test when possible
#

#  load manifest and detect columns robustly
if not Path(MANIFEST).exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")
df = pd.read_csv(MANIFEST)
print("Manifest rows:", len(df))
print("Manifest columns:", df.columns.tolist())

# candidate column names
candidate_utt_cols = ["utt_id","utt","id","filename","file","wav","wav_name","wav_path","path","audio_path","audio"]
candidate_speaker_cols = ["speaker_id","speaker","spk","speakerID","spk_id"]
candidate_label_cols = ["label","lang","language","target"]

utt_col = next((c for c in candidate_utt_cols if c in df.columns), None)
spk_col = next((c for c in candidate_speaker_cols if c in df.columns), None)
lab_col = next((c for c in candidate_label_cols if c in df.columns), None)
print("Detected columns -> utt_col:", utt_col, "spk_col:", spk_col, "lab_col:", lab_col)
if lab_col is None:
    raise KeyError("No label column detected in manifest. Edit manifest or update candidate_label_cols.")

def get_utt_from_row(row):
    # return file stem suitable to match .npz stems
    if utt_col and utt_col in df.columns and pd.notna(row.get(utt_col)):
        val = row.get(utt_col)
        p = Path(str(val))
        return p.stem if p.suffix else str(val)
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).stem
    return str(row.name)

def get_spk_from_row(row):
    if spk_col and spk_col in df.columns and pd.notna(row.get(spk_col)):
        return str(row.get(spk_col))
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).parent.name
    return "unknown"

#  index hubert .npz files once (fast)
hubdir = Path(HUBERT_DIR)
if not hubdir.exists():
    raise FileNotFoundError(f"HuBERT dir not found: {HUBERT_DIR}")
hubert_index = {p.stem: p for p in hubdir.rglob("*.npz")}
print("Indexed .npz files:", len(hubert_index))

#  collect pooled arrays (aligned to utts present)
utt_to_pooled = {}
utts_collected = []
n_layers_detected = None
skipped = 0
for _, row in df.iterrows():
    utt = get_utt_from_row(row)
    p = hubert_index.get(utt) or hubert_index.get(utt.replace(" ", "_")) or hubert_index.get(utt.replace("_"," "))
    if not p:
        skipped += 1
        continue
    try:
        data = np.load(p)
    except Exception as e:
        skipped += 1
        continue
    pooled = None
    if "pooled" in data:
        pooled = data["pooled"]
    elif "hidden_states" in data:
        # hidden_states may be array/list of layers -> pool tokens per layer
        try:
            pooled = np.array([h.mean(axis=0) for h in data["hidden_states"]])
        except:
            pooled = None
    else:
        arrays = [data[k] for k in data.files]
        # prefer first 2D array (layers x dim)
        chosen = next((a for a in arrays if getattr(a,'ndim',0)==2), None)
        if chosen is not None:
            pooled = chosen
        elif len(arrays)==1 and arrays[0].ndim==1:
            pooled = arrays[0][None,:]
    if pooled is None:
        skipped += 1
        continue
    if n_layers_detected is None:
        n_layers_detected = pooled.shape[0] if pooled.ndim==2 else 1
    # accept only if matching detected layers
    if pooled.ndim==2 and pooled.shape[0]==n_layers_detected:
        utt_to_pooled[utt] = pooled
        utts_collected.append(utt)
    else:
        skipped += 1

print("Collected pooled utts:", len(utts_collected), "skipped:", skipped, "detected_layers:", n_layers_detected)
if len(utts_collected) == 0:
    raise RuntimeError("No usable HuBERT pooled files found. Check stems and manifest.")

#  quick per-layer scan (speaker-wise) to pick best layer if no saved results
best_layer = None
if Path("layerwise_results.json").exists():
    try:
        with open("layerwise_results.json","r") as f:
            saved = json.load(f)
        if isinstance(saved, dict) and saved:
            saved_int = {int(k): float(v) for k,v in saved.items()}
            best_layer = max(saved_int, key=saved_int.get)
            print("Loaded best_layer from layerwise_results.json:", best_layer, "acc:", saved_int[best_layer])
    except Exception:
        best_layer = None

if best_layer is None:
    print("No cached layer results -> performing safe quick per-layer scan.")
    # build labels and groups aligned to utts_collected
    labels_all = []
    groups_all = []
    for utt in utts_collected:
        row = df[df.apply(lambda r: get_utt_from_row(r)==utt, axis=1)]
        if row.shape[0]==0:
            labels_all.append(None); groups_all.append("unknown")
        else:
            labels_all.append(str(row.iloc[0][lab_col]))
            groups_all.append(get_spk_from_row(row.iloc[0]))
    labels_arr = np.array(labels_all)
    groups_arr = np.array(groups_all)
    # encode labels (drop None)
    valid_mask = labels_arr != None
    if valid_mask.sum() == 0:
        raise RuntimeError("No labels found for collected utts.")
    labels_valid = labels_arr[valid_mask]
    groups_valid = groups_arr[valid_mask]
    utts_valid = np.array(utts_collected)[valid_mask]
    le_tmp = LabelEncoder().fit(labels_valid)
    y_enc = le_tmp.transform(labels_valid)
    layer_scores = {}
    # iterate layers
    for l in range(n_layers_detected):
        Xl = np.vstack([utt_to_pooled[u][l] for u in utts_valid])
        # scale + small PCA
        scalerL = StandardScaler().fit(Xl)
        Xl_s = scalerL.transform(Xl)
        n_comp = min(64, Xl_s.shape[1], Xl_s.shape[0]-1)
        if n_comp <= 0:
            layer_scores[l] = 0.0
            print(f"Layer {l:02d} quick-acc: 0.0 (not enough samples/features)")
            continue
        pcaL = PCA(n_components=n_comp, random_state=RANDOM_STATE).fit(Xl_s)
        Xl_p = pcaL.transform(Xl_s)
        # speaker-wise split with retry (small number of tries here)
        acc_val = 0.0
        success = False
        for attempt_seed in range(5):
            gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE+attempt_seed)
            try:
                tr, te = next(gss.split(Xl_p, y_enc, groups_valid))
            except Exception:
                continue
            if len(np.unique(y_enc[te])) < 2:
                continue
            clf = LogisticRegression(max_iter=2000, multi_class='multinomial', solver='saga')
            clf.fit(Xl_p[tr], y_enc[tr])
            acc_val = accuracy_score(y_enc[te], clf.predict(Xl_p[te]))
            success = True
            break
        layer_scores[l] = float(acc_val) if success else 0.0
        print(f"Layer {l:02d} quick-acc: {layer_scores[l]:.4f}")
    # pick best
    best_layer = max(layer_scores, key=layer_scores.get)
    print("Picked best layer:", best_layer, "acc:", layer_scores[best_layer])
    # save quick layer scores
    with open("layerwise_results.json","w") as f:
        json.dump(layer_scores, f)

# build dataset using best_layer vectors (one vector per utt)
X_list, y_list, groups = [], [], []
for _, row in df.iterrows():
    utt = get_utt_from_row(row)
    p = hubert_index.get(utt) or hubert_index.get(utt.replace(" ", "_"))
    if not p:
        continue
    data = np.load(p)
    pooled = None
    if "pooled" in data:
        pooled = data["pooled"]
    elif "hidden_states" in data:
        pooled = np.array([h.mean(axis=0) for h in data["hidden_states"]])
    else:
        arrays = [data[k] for k in data.files]
        chosen = next((a for a in arrays if getattr(a,'ndim',0)==2), None)
        if chosen is not None:
            pooled = chosen
        elif len(arrays)==1 and arrays[0].ndim==1:
            pooled = arrays[0][None,:]
    if pooled is None:
        continue
    if pooled.ndim==2 and pooled.shape[0] > best_layer:
        vec = pooled[best_layer]
    else:
        vec = pooled.mean(axis=0) if pooled.ndim==2 else pooled
    X_list.append(vec)
    y_list.append(str(row[lab_col]))
    groups.append(get_spk_from_row(row))

X = np.vstack(X_list)
y = np.array(y_list)
groups = np.array(groups)
print("Built dataset shape:", X.shape, "label counts:", Counter(y))

#  filter rare classes (MIN_UTTS)
counts = Counter(y)
keep = {lab for lab,c in counts.items() if c >= MIN_UTTS}
print("Original classes:", len(counts), "kept classes (>= {} utts): {}".format(MIN_UTTS, len(keep)))
if len(keep) < 2:
    raise RuntimeError("Too few classes after applying MIN_UTTS. Lower MIN_UTTS or collect more data.")
Xf = []
yf = []
gf = []
for xi, yi, gi in zip(X, y, groups):
    if yi in keep:
        Xf.append(xi); yf.append(yi); gf.append(gi)
Xf = np.vstack(Xf); yf = np.array(yf); gf = np.array(gf)
print("After filtering:", Xf.shape, Counter(yf))

#  label encode & robust speaker-wise split with retry
le = LabelEncoder().fit(yf)
y_enc = le.transform(yf)
unique_labels = np.unique(y_enc)

def find_speakerwise_split(Xa, ya, groups_arr, max_tries=MAX_SPLIT_TRIES, min_per_class=MIN_PER_CLASS_IN_TEST):
    rng = np.random.RandomState(RANDOM_STATE)
    for t in range(max_tries):
        seed = rng.randint(0, 2**31-1)
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=int(seed))
        tr, te = next(gss.split(Xa, ya, groups_arr))
        labels_in_test, counts_test = np.unique(ya[te], return_counts=True)
        ok = True
        for lab in unique_labels:
            # ensure label present in test with at least min_per_class
            cnt = counts_test[labels_in_test.tolist().index(lab)] if lab in labels_in_test else 0
            if cnt < min_per_class:
                ok = False; break
        if ok:
            return tr, te, seed
    return None, None, None

tr, te, used_seed = find_speakerwise_split(Xf, y_enc, gf)
if tr is None:
    # second attempt: allow at least 1 sample per class in test
    tr, te, used_seed = find_speakerwise_split(Xf, y_enc, gf, max_tries=200, min_per_class=1)
if tr is None:
    # fallback: utterance-level stratified split (may leak speakers)
    print("Warning: couldn't find suitable speaker-wise split that preserves classes -> falling back to utterance-level stratified split.")
    tr_idx, te_idx = train_test_split(np.arange(len(y_enc)), test_size=TEST_SIZE, stratify=y_enc, random_state=RANDOM_STATE)
    tr, te = tr_idx, te_idx
    used_seed = "stratified_fallback"

print("Split seed used:", used_seed)
print("Train/test sizes:", len(tr), len(te))
print("Train label counts:", Counter(y_enc[tr]))
print("Test label counts:", Counter(y_enc[te]))

Xtr, Xte = Xf[tr], Xf[te]
ytr, yte = y_enc[tr], y_enc[te]

#  scale, PCA (safe sizing), train RF
scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr); Xte_s = scaler.transform(Xte)
max_components = min(256, Xtr_s.shape[1], max(1, Xtr_s.shape[0]-1))
if max_components <= 1:
    print("Skipping PCA (not enough samples/features).")
    Xtr_p, Xte_p = Xtr_s, Xte_s
else:
    pca = PCA(n_components=max_components, random_state=RANDOM_STATE).fit(Xtr_s)
    Xtr_p, Xte_p = pca.transform(Xtr_s), pca.transform(Xte_s)
    print("PCA applied with n_components:", max_components)

clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
clf.fit(Xtr_p, ytr)
ypred = clf.predict(Xte_p)
acc = accuracy_score(yte, ypred)
print(f"HuBERT best-layer ({best_layer}) speaker-wise accuracy: {acc:.4f}")

# safe classification_report (only for present labels in test)
present_labels = sorted(list(set(ytr).union(set(yte))))
present_names = [le.classes_[i] for i in present_labels]
print("Classification report (present labels only):")
print(classification_report(yte, ypred, labels=present_labels, target_names=present_names, zero_division=0))

# confusion matrix
cm = confusion_matrix(yte, ypred, labels=present_labels)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=present_names, yticklabels=present_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"HuBERT layer {best_layer} - Confusion Matrix")
plt.show()

# Save artifacts
artifacts = {
    "clf": clf,
    "scaler": scaler,
    "label_encoder": le,
    "best_layer": int(best_layer),
    "classes_kept": list(sorted(keep)),
    "split_seed": used_seed
}
# save PCA if used
if 'pca' in locals():
    artifacts['pca'] = pca
joblib.dump(artifacts, MODEL_DIR / "hubert_bestlayer_rf_safe.joblib")
with open(MODEL_DIR / "hubert_training_info.json", "w") as f:
    json.dump({
        "best_layer": int(best_layer),
        "n_collected_utts": len(utts_collected),
        "n_used_utts": int(Xf.shape[0]),
        "classes_kept": list(sorted(keep)),
        "train_size": int(len(tr)),
        "test_size": int(len(te)),
        "split_seed": str(used_seed),
        "acc": float(acc)
    }, f, indent=2)
print("Saved model & info to", MODEL_DIR)

In [ ]:
#  Train on sample (existing .npz files) and produce report
import os, json
from pathlib import Path
import numpy as np, pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# CONFIG - change if needed
MANIFEST = "metadata.csv"
HUBERT_DIR = "features/hubert"
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR = Path("reports"); REPORT_DIR.mkdir(exist_ok=True)
REPORT_MD = REPORT_DIR / "hubert_sample_report.md"
TEST_SIZE = 0.20
RANDOM_STATE = 42
MIN_UTTS = 6                # filter tiny classes (tuneable)
MIN_PER_CLASS_IN_TEST = 1   # require at least 1 sample per class in test when possible
MAX_SPLIT_TRIES = 300

# Load manifest
df = pd.read_csv(MANIFEST)
print("Manifest rows:", len(df))

# Build hubert index (use only existing .npz files)
hubdir = Path(HUBERT_DIR)
if not hubdir.exists():
    raise FileNotFoundError(f"HuBERT dir not found: {HUBERT_DIR}")
hubert_files = sorted([p for p in hubdir.rglob("*.npz")])
print("Found .npz files:", len(hubert_files))
hubert_index = {p.stem: p for p in hubert_files}

# helper to extract utt stem and speaker (robust)
candidate_utt_cols = ["utt_id","utt","id","filename","file","wav","wav_name","wav_path","path","audio_path","audio"]
candidate_speaker_cols = ["speaker_id","speaker","spk","speakerID","spk_id"]
candidate_label_cols = ["label","lang","language","target"]

utt_col = next((c for c in candidate_utt_cols if c in df.columns), None)
spk_col = next((c for c in candidate_speaker_cols if c in df.columns), None)
lab_col = next((c for c in candidate_label_cols if c in df.columns), None)
if lab_col is None:
    raise KeyError("No label column detected in manifest. Edit manifest or candidate_label_cols.")

def get_utt_from_row(row):
    if utt_col and utt_col in df.columns and pd.notna(row.get(utt_col)):
        p = Path(str(row.get(utt_col)))
        return p.stem if p.suffix else str(row.get(utt_col))
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).stem
    return str(row.name)

def get_spk_from_row(row):
    if spk_col and spk_col in df.columns and pd.notna(row.get(spk_col)):
        return str(row.get(spk_col))
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).parent.name
    return "unknown"

# Collect only utts present in hubert_index (sample)
X_list, y_list, groups = [], [], []
collected_utts = []
for _, row in df.iterrows():
    utt = get_utt_from_row(row)
    p = hubert_index.get(utt) or hubert_index.get(utt.replace(" ", "_")) or hubert_index.get(utt.replace("_"," "))
    if not p:
        continue
    data = np.load(p)
    pooled = None
    if "pooled" in data:
        pooled = data["pooled"]
    elif "hidden_states" in data:
        pooled = np.array([h.mean(axis=0) for h in data["hidden_states"]])
    else:
        arrays = [data[k] for k in data.files]
        chosen = next((a for a in arrays if getattr(a,'ndim',0)==2), None)
        if chosen is not None:
            pooled = chosen
        elif len(arrays)==1 and arrays[0].ndim==1:
            pooled = arrays[0][None,:]
    if pooled is None:
        continue
    # choose best representation per utt: mean across layers (fallback). If you have layerwise_results.json prefer that later.
    collected_utts.append(utt)
    X_list.append(pooled)   # store pooled (layers x dim) for flexibility
    y_list.append(str(row[lab_col]))
    groups.append(get_spk_from_row(row))

n_sample = len(X_list)
print(f"Collected {n_sample} utts present in {HUBERT_DIR} (using as sample).")
if n_sample == 0:
    raise RuntimeError("No .npz matched the manifest - check stems.")

# If layerwise_results.json exists and contains a best layer, use it; otherwise do a quick layer scan using only this sample
best_layer = None
if Path("layerwise_results.json").exists():
    try:
        with open("layerwise_results.json","r") as f:
            saved = json.load(f)
        saved_int = {int(k): float(v) for k,v in saved.items()}
        best_layer = max(saved_int, key=saved_int.get)
        print("Picked best_layer from layerwise_results.json:", best_layer, "acc:", saved_int[best_layer])
    except Exception:
        best_layer = None

# Quick per-layer scan on the sample to pick best layer (safe, limited tries)
if best_layer is None:
    print("No saved best_layer -> quick per-layer scan on sample")
    # build per-layer arrays
    # ensure all pooled shapes consistent
    if not all(getattr(a,'ndim',0)==2 for a in X_list):
        # if mixing shapes, convert each to (layers, dim) or fallback to mean vector
        X_mean = [ (a.mean(axis=0) if a.ndim==2 else a) for a in X_list ]
        # save mean vectors to use as single-layer fallback
        pooled_list = None
        n_layers = 1
    else:
        n_layers = X_list[0].shape[0]
        pooled_list = X_list
    # prepare labels and groups arrays aligned
    labels_arr = np.array(y_list)
    groups_arr = np.array(groups)
    # label encode
    le_tmp = LabelEncoder().fit(labels_arr)
    y_enc_tmp = le_tmp.transform(labels_arr)
    layer_scores = {}
    from sklearn.linear_model import LogisticRegression
    for l in range(n_layers):
        if pooled_list is None:
            Xl = np.vstack([v if v.ndim==1 else v for v in X_mean])  # mean vectors
        else:
            Xl = np.vstack([p[l] for p in pooled_list])
        # scale + small pca
        scalerL = StandardScaler().fit(Xl)
        Xl_s = scalerL.transform(Xl)
        n_comp = min(64, Xl_s.shape[1], max(1, Xl_s.shape[0]-1))
        if n_comp <= 0:
            layer_scores[l] = 0.0; print(f"Layer {l} quick-acc: 0.0 (too few samples)"); continue
        pcaL = PCA(n_components=n_comp, random_state=RANDOM_STATE).fit(Xl_s)
        Xl_p = pcaL.transform(Xl_s)
        # speaker-wise split but only try a few seeds to ensure test has >1 class
        acc_val = 0.0; success=False
        for seed_offset in range(6):
            gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE+seed_offset)
            try:
                tr, te = next(gss.split(Xl_p, y_enc_tmp, groups_arr))
            except:
                continue
            if len(np.unique(y_enc_tmp[te])) < 2:
                continue
            clf = LogisticRegression(max_iter=2000, multi_class='multinomial', solver='saga')
            clf.fit(Xl_p[tr], y_enc_tmp[tr])
            acc_val = accuracy_score(y_enc_tmp[te], clf.predict(Xl_p[te]))
            success=True; break
        layer_scores[l] = float(acc_val) if success else 0.0
        print(f"Layer {l:02d} quick-acc: {layer_scores[l]:.4f}")
    best_layer = max(layer_scores, key=layer_scores.get)
    print("Selected best_layer (on sample):", best_layer, "acc:", layer_scores[best_layer])
    with open("layerwise_results.json","w") as f:
        json.dump(layer_scores, f)

# Build final dataset using best_layer (or mean across layers if pooled is 1D)
X_vecs, y_vecs, groups_vecs = [], [], []
for pooled, lab, spk in zip(X_list, y_list, groups):
    if getattr(pooled,'ndim',0)==2 and pooled.shape[0] > best_layer:
        vec = pooled[best_layer]
    elif getattr(pooled,'ndim',0)==2:
        vec = pooled.mean(axis=0)
    else:
        vec = pooled
    X_vecs.append(vec); y_vecs.append(lab); groups_vecs.append(spk)

X = np.vstack(X_vecs); y = np.array(y_vecs); groups = np.array(groups_vecs)
print("Final dataset shape (used for training):", X.shape, "label counts:", Counter(y))

# Filter rare classes (MIN_UTTS)
counts = Counter(y)
keep = {lab for lab,c in counts.items() if c >= MIN_UTTS}
print("Classes before filter:", len(counts), "after filter (>= {} utts): {}".format(MIN_UTTS, len(keep)))
Xf, yf, gf = [], [], []
for xi, yi, gi in zip(X, y, groups):
    if yi in keep:
        Xf.append(xi); yf.append(yi); gf.append(gi)
if len(Xf) == 0:
    raise RuntimeError("No data left after filtering. Lower MIN_UTTS or choose another approach.")
Xf = np.vstack(Xf); yf = np.array(yf); gf = np.array(gf)
print("Filtered dataset:", Xf.shape, Counter(yf))

# Label encode
le = LabelEncoder().fit(yf)
y_enc = le.transform(yf)

# Robust speaker-wise split with retries
def find_good_split(Xa, ya, groups_arr, tries=MAX_SPLIT_TRIES, min_per_class=MIN_PER_CLASS_IN_TEST):
    rng = np.random.RandomState(RANDOM_STATE)
    unique_labels = np.unique(ya)
    for t in range(tries):
        seed = int(rng.randint(0,2**31-1))
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        tr, te = next(gss.split(Xa, ya, groups_arr))
        labs_te, counts_te = np.unique(ya[te], return_counts=True)
        ok = True
        for lab in unique_labels:
            if lab not in labs_te:
                ok = False; break
            # optionally check counts_te[...] >= min_per_class
            if counts_te[labs_te.tolist().index(lab)] < min_per_class:
                ok = False; break
        if ok:
            return tr, te, seed
    return None, None, None

tr, te, used_seed = find_good_split(Xf, y_enc, gf)
if tr is None:
    print("Could not find perfect speaker-wise split that preserves all classes; retrying with min_per_class=1")
    tr, te, used_seed = find_good_split(Xf, y_enc, gf, tries=200, min_per_class=1)
if tr is None:
    print("Falling back to utterance-level stratified split (possible speaker leakage).")
    tr_idx, te_idx = train_test_split(np.arange(len(y_enc)), test_size=TEST_SIZE, stratify=y_enc, random_state=RANDOM_STATE)
    tr, te = tr_idx, te_idx; used_seed = "stratified_fallback"

print("Split used seed:", used_seed)
print("Train/test sizes:", len(tr), len(te))
print("Train label counts:", Counter(y_enc[tr]))
print("Test label counts:", Counter(y_enc[te]))

Xtr, Xte = Xf[tr], Xf[te]
ytr, yte = y_enc[tr], y_enc[te]

# Scale and PCA safely
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
max_comp = min(256, Xtr_s.shape[1], max(1, Xtr_s.shape[0]-1))
if max_comp <= 1:
    Xtr_p, Xte_p = Xtr_s, Xte_s
    pca_used = False
else:
    pca = PCA(n_components=max_comp, random_state=RANDOM_STATE).fit(Xtr_s)
    Xtr_p, Xte_p = pca.transform(Xtr_s), pca.transform(Xte_s)
    pca_used = True
    print("Applied PCA n_components:", max_comp)

# Train RF
clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
clf.fit(Xtr_p, ytr)
ypred = clf.predict(Xte_p)

acc = accuracy_score(yte, ypred)
print(f"Sample HuBERT best-layer ({best_layer}) speaker-wise accuracy: {acc:.4f}")

# report metrics
present_labels = sorted(list(set(ytr).union(set(yte))))
present_names = [le.classes_[i] for i in present_labels]
report_text = []
report_text.append("# HuBERT sample experiment report\n")
report_text.append(f"**Date:** {pd.Timestamp.now()}\n\n")
report_text.append("## Summary\n")
report_text.append(f"- Used sample of existing HuBERT .npz files in `{HUBERT_DIR}`: **{len(hubert_files)} files found**, **{n_sample} matched manifest entries**, **{Xf.shape[0]} used after class filtering**.\n")
report_text.append(f"- Best HuBERT layer used: **{best_layer}** (auto-selected from quick scan or cached results).\n")
report_text.append(f"- Model: RandomForest (n_estimators=300), speaker-wise test size: {TEST_SIZE}\n")
report_text.append(f"- Speaker-wise test accuracy: **{acc:.4f}**\n\n")

report_text.append("## Class distribution (used)\n")
for lab, cnt in Counter(yf).most_common():
    report_text.append(f"- {lab}: {cnt} utts\n")
report_text.append("\n")

report_text.append("## Limitations & notes\n")
report_text.append("- This experiment uses only the EXISTING sample of HuBERT `.npz` files (no further extraction). It is NOT the full dataset (we had more audio files in the manifest). Results are **sample-based** and may not generalize to the full dataset.\n")
report_text.append("- Small classes were filtered out with `MIN_UTTS={}` to avoid unstable training. Consider collecting more data or merging rare classes for a full evaluation.\n".format(MIN_UTTS))
report_text.append("- Evaluation uses a speaker-wise split when possible. If an exact speaker-wise split preserving all classes was not possible, a stratified utterance-level split was used as fallback — this is noted in saved metadata.\n\n")

report_text.append("## Classification report (present labels only)\n")
report_text.append("```\n")
report_text.append(classification_report(yte, ypred, labels=present_labels, target_names=present_names, zero_division=0))
report_text.append("\n```\n")

# confusion matrix plot and save
cm = confusion_matrix(yte, ypred, labels=present_labels)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=present_names, yticklabels=present_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"HuBERT layer {best_layer} - Confusion Matrix (sample)")
plt.tight_layout()
plt.show()

# Save artifacts and report
artifacts = {"clf": clf, "scaler": scaler, "label_encoder": le, "best_layer": int(best_layer), "classes_kept": list(sorted(keep)), "split_seed": str(used_seed)}
if pca_used:
    artifacts["pca"] = pca
joblib.dump(artifacts, MODEL_DIR / "hubert_bestlayer_sample_rf.joblib")
with open(MODEL_DIR / "hubert_sample_info.json","w") as f:
    json.dump({
        "n_found_npz": len(hubert_files),
        "n_matched_manifest": n_sample,
        "n_used_after_filter": int(Xf.shape[0]),
        "best_layer": int(best_layer),
        "acc": float(acc),
        "split_seed": str(used_seed)
    }, f, indent=2)

# write markdown report
with open(REPORT_MD, "w") as f:
    f.write("\n".join(report_text))
print("Saved model to", MODEL_DIR / "hubert_bestlayer_sample_rf.joblib")
print("Saved report to", REPORT_MD)
print("Saved training info to", MODEL_DIR / "hubert_sample_info.json")

# show top of report
print("\n--- Report preview ---\n")
print("\n".join(report_text[:20]))

In [ ]:
#  Step-1 completion: MFCC baseline + MFCC vs HuBERT comparison
import os, json
from pathlib import Path
import numpy as np, pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

#  CONFIG
MANIFEST = "metadata.csv"
MFCC_DIR = "features/mfcc"           # directory containing per-utt .npy MFCC files
HUBERT_INFO = Path("models/hubert_sample_info.json")  # optional: contains hubert sample acc & split info
HUBERT_MODEL = Path("models/hubert_bestlayer_sample_rf.joblib")
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR = Path("reports"); REPORT_DIR.mkdir(exist_ok=True)
MFCC_REPORT_MD = REPORT_DIR / "mfcc_baseline_report.md"
COMPARE_REPORT_MD = REPORT_DIR / "baseline_comparison.md"

TEST_SIZE = 0.20
RANDOM_STATE = 42
MIN_UTTS = 6   # same filter rule used for HuBERT sample (tuneable)


# Load manifest
if not Path(MANIFEST).exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")
df = pd.read_csv(MANIFEST)
print("Manifest rows:", len(df))

# detect columns robustly
candidate_utt_cols = ["utt_id","utt","id","filename","file","wav","wav_name","wav_path","path","audio_path","audio"]
candidate_speaker_cols = ["speaker_id","speaker","spk","speakerID","spk_id"]
candidate_label_cols = ["label","lang","language","target"]

utt_col = next((c for c in candidate_utt_cols if c in df.columns), None)
spk_col = next((c for c in candidate_speaker_cols if c in df.columns), None)
lab_col = next((c for c in candidate_label_cols if c in df.columns), None)
if lab_col is None:
    raise KeyError("No label column detected in manifest. Edit manifest or candidate_label_cols.")

def get_utt_from_row(row):
    if utt_col and utt_col in df.columns and pd.notna(row.get(utt_col)):
        p = Path(str(row.get(utt_col)))
        return p.stem if p.suffix else str(row.get(utt_col))
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).stem
    return str(row.name)

def get_spk_from_row(row):
    if spk_col and spk_col in df.columns and pd.notna(row.get(spk_col)):
        return str(row.get(spk_col))
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).parent.name
    return "unknown"

# Build MFCC index (fast)
mfcc_dir = Path(MFCC_DIR)
if not mfcc_dir.exists():
    raise FileNotFoundError(f"MFCC dir not found: {MFCC_DIR}")
mfcc_index = {p.stem: p for p in mfcc_dir.rglob("*.npy")}
print("Indexed MFCC .npy files:", len(mfcc_index))

# Load hubert info if present (to match classes kept and get hubert acc)
hubert_info = {}
if HUBERT_INFO.exists():
    try:
        hubert_info = json.loads(HUBERT_INFO.read_text())
        print("Loaded hubert sample info:", HUBERT_INFO)
    except Exception:
        hubert_info = {}
else:
    print("No hubert_sample_info.json found; will still run MFCC baseline and compare only if hubert model exists.")

# Determine classes-to-keep
classes_kept = None
if 'classes_kept' in hubert_info and hubert_info['classes_kept']:
    classes_kept = set(hubert_info['classes_kept'])
    print("Using classes_kept from hubert_info:", len(classes_kept))
else:
    # derive keep labels from MFCC counts (>= MIN_UTTS)
    # We'll compute raw counts first then filter after collecting MFCCs
    classes_kept = None

# Collect MFCC features aligned to manifest rows and keep only utts with MFCC files
X_mfcc_list = []
y_mfcc_list = []
groups_mfcc = []
utts_mfcc = []
for _, row in df.iterrows():
    utt = get_utt_from_row(row)
    p = mfcc_index.get(utt) or mfcc_index.get(utt.replace(" ", "_")) or mfcc_index.get(utt.replace("_"," "))
    if not p:
        continue
    try:
        arr = np.load(p)
    except Exception:
        continue
    # pool mean across time if 2D
    if getattr(arr, "ndim", 0) == 2:
        vec = arr.mean(axis=1)
    else:
        vec = arr
    X_mfcc_list.append(vec)
    lab = str(row[lab_col])
    y_mfcc_list.append(lab)
    groups_mfcc.append(get_spk_from_row(row))
    utts_mfcc.append(utt)

n_mfcc = len(X_mfcc_list)
print(f"Collected {n_mfcc} MFCC files matching manifest.")

if n_mfcc == 0:
    raise RuntimeError("No MFCC files matched the manifest. Check MFCC_DIR and stems.")

# Convert to arrays
X_mfcc = np.vstack(X_mfcc_list)
y_mfcc = np.array(y_mfcc_list)
groups_mfcc = np.array(groups_mfcc)

# If classes_kept known from hubert_info, filter MFCC dataset to those classes (so MFCC vs HuBERT comparable)
if classes_kept is not None:
    mask = np.array([lab in classes_kept for lab in y_mfcc])
    X_mfcc = X_mfcc[mask]; y_mfcc = y_mfcc[mask]; groups_mfcc = groups_mfcc[mask]
    print("After filtering to hubert-kept classes, MFCC dataset shape:", X_mfcc.shape)
else:
    # otherwise filter rare classes using MIN_UTTS to avoid unstable classes
    cnts = Counter(y_mfcc)
    keep = {lab for lab,c in cnts.items() if c >= MIN_UTTS}
    print("MFCC class counts:", cnts)
    print("Keeping classes with >= {} utts: {}".format(MIN_UTTS, len(keep)))
    mask = np.array([lab in keep for lab in y_mfcc])
    X_mfcc = X_mfcc[mask]; y_mfcc = y_mfcc[mask]; groups_mfcc = groups_mfcc[mask]
    print("After filtering MFCC dataset shape:", X_mfcc.shape)

# label encode
le_mfcc = LabelEncoder().fit(y_mfcc)
y_mfcc_enc = le_mfcc.transform(y_mfcc)
print("MFCC label classes:", list(le_mfcc.classes_))

# Try to use same split seed as hubert (if available) to be as comparable as possible
split_seed = RANDOM_STATE
if 'split_seed' in hubert_info:
    try:
        seed_val = hubert_info['split_seed']
        # if stored as numeric string convert to int; if "stratified_fallback" ignore
        if isinstance(seed_val, str) and seed_val.isdigit():
            split_seed = int(seed_val)
        elif isinstance(seed_val, int):
            split_seed = int(seed_val)
        else:
            # leave default
            split_seed = RANDOM_STATE
    except Exception:
        split_seed = RANDOM_STATE

# Robust speaker-wise split (try to ensure test contains multiple classes)
def find_speakerwise_split(Xa, ya, groups_arr, test_size=TEST_SIZE, tries=500, min_per_class=1, seed_base=RANDOM_STATE):
    rng = np.random.RandomState(seed_base)
    unique = np.unique(ya)
    for t in range(tries):
        seed = int(rng.randint(0,2**31-1))
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        tr, te = next(gss.split(Xa, ya, groups_arr))
        labs_te, counts_te = np.unique(ya[te], return_counts=True)
        ok = True
        for lab in unique:
            if lab not in labs_te:
                ok=False; break
            if counts_te[labs_te.tolist().index(lab)] < min_per_class:
                ok=False; break
        if ok:
            return tr, te, seed
    return None, None, None

tr, te, used_seed = find_speakerwise_split(X_mfcc, y_mfcc_enc, groups_mfcc, test_size=TEST_SIZE, tries=500, min_per_class=1, seed_base=split_seed)
if tr is None:
    print("Could not find ideal speaker-wise split preserving classes -> falling back to utterance stratified split (may leak speakers).")
    tr_idx, te_idx = train_test_split(np.arange(len(y_mfcc_enc)), test_size=TEST_SIZE, stratify=y_mfcc_enc, random_state=split_seed)
    tr, te = tr_idx, te_idx
    used_seed = "stratified_fallback"

print("Using split seed:", used_seed)
print("MFCC Train/Test sizes:", len(tr), len(te))
print("MFCC Train label counts:", Counter(y_mfcc_enc[tr]))
print("MFCC Test label counts:", Counter(y_mfcc_enc[te]))

Xtr, Xte = X_mfcc[tr], X_mfcc[te]
ytr, yte = y_mfcc_enc[tr], y_mfcc_enc[te]

# scale + safe PCA + RF training
scaler_m = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler_m.transform(Xtr), scaler_m.transform(Xte)
max_comp = min(256, Xtr_s.shape[1], max(1, Xtr_s.shape[0]-1))
if max_comp <= 1:
    Xtr_p, Xte_p = Xtr_s, Xte_s
    pca_used = False
else:
    pca_m = PCA(n_components=max_comp, random_state=RANDOM_STATE).fit(Xtr_s)
    Xtr_p, Xte_p = pca_m.transform(Xtr_s), pca_m.transform(Xte_s)
    pca_used = True
    print("MFCC PCA applied n_components:", max_comp)

clf_mfcc = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
clf_mfcc.fit(Xtr_p, ytr)
yp_mfcc = clf_mfcc.predict(Xte_p)

acc_mfcc = accuracy_score(yte, yp_mfcc)
print("MFCC speaker-wise accuracy:", acc_mfcc)
print("MFCC classification report:")
present_labels = sorted(list(set(ytr).union(set(yte))))
present_names = [le_mfcc.classes_[i] for i in present_labels]
print(classification_report(yte, yp_mfcc, labels=present_labels, target_names=present_names, zero_division=0))

# Save MFCC artifacts
joblib.dump({"clf": clf_mfcc, "scaler": scaler_m, "pca": (pca_m if pca_used else None), "label_encoder": le_mfcc}, MODEL_DIR / "mfcc_baseline_rf.joblib")
with open(MODEL_DIR / "mfcc_baseline_info.json", "w") as f:
    json.dump({
        "acc": float(acc_mfcc),
        "n_used": int(X_mfcc.shape[0]),
        "n_after_filter": int(Xtr.shape[0] + Xte.shape[0]),
        "classes_kept": list(le_mfcc.classes_),
        "split_seed": str(used_seed)
    }, f, indent=2)
print("Saved MFCC model and info to", MODEL_DIR)

# Attempt to obtain HuBERT accuracy from hubert info or model
hubert_acc = None
hubert_used = 0
hubert_classes = None
if HUBERT_INFO.exists():
    try:
        info = json.loads(HUBERT_INFO.read_text())
        hubert_acc = float(info.get("acc", None)) if info.get("acc", None) is not None else None
        hubert_used = int(info.get("n_used_after_filter", info.get("n_matched_manifest", 0)))
        hubert_classes = info.get("classes_kept", None)
    except Exception:
        hubert_acc = None

# If no info, try to load hubert model and evaluate on its saved test split (if available) - best-effort
if hubert_acc is None and HUBERT_MODEL.exists():
    try:
        art = joblib.load(HUBERT_MODEL)
        # art expected to contain 'clf','scaler','pca','label_encoder','best_layer'
        print("Loaded hubert model for best-effort lookup.")
        # no straightforward access to hubert test split here; skip detailed evaluation
    except Exception:
        pass

# Build comparison report text
cmp_lines = []
cmp_lines.append("# Baseline comparison: MFCC vs HuBERT (sample)\n")
cmp_lines.append(f"- Date: {pd.Timestamp.now()}\n")
cmp_lines.append("## Summary\n")
cmp_lines.append(f"- MFCC dataset: matched MFCC files = {n_mfcc}; used after filtering = {Xtr.shape[0]+Xte.shape[0]}\n")
if hubert_used:
    cmp_lines.append(f"- HuBERT sample: used {hubert_used} utts (sample-based extraction).\n")
else:
    cmp_lines.append(f"- HuBERT sample: info not found; please run HuBERT sample cell first.\n")
cmp_lines.append(f"- MFCC speaker-wise test accuracy: **{acc_mfcc:.4f}**\n")
if hubert_acc is not None:
    cmp_lines.append(f"- HuBERT sample speaker-wise accuracy (from saved info): **{hubert_acc:.4f}**\n")
else:
    cmp_lines.append(f"- HuBERT sample accuracy: **(not available)**\n")
cmp_lines.append("\n## MFCC classification report (present labels)\n")
cmp_lines.append("```\n")
cmp_lines.append(classification_report(yte, yp_mfcc, labels=present_labels, target_names=present_names, zero_division=0))
cmp_lines.append("\n```\n")

# Save MFCC report
with open(MFCC_REPORT_MD, "w") as f:
    f.write("# MFCC Baseline Report\n\n")
    f.write(f"Date: {pd.Timestamp.now()}\n\n")
    f.write(f"MFCC files matched manifest: {n_mfcc}\n\n")
    f.write(f"MFCC speaker-wise accuracy: {acc_mfcc:.4f}\n\n")
    f.write("Classification report (present labels):\n\n")
    f.write("```\n")
    f.write(classification_report(yte, yp_mfcc, labels=present_labels, target_names=present_names, zero_division=0))
    f.write("\n```\n")
print("Saved MFCC report to", MFCC_REPORT_MD)

# Save comparison report
with open(COMPARE_REPORT_MD, "w") as f:
    f.write("\n".join(cmp_lines))
print("Saved comparison report to", COMPARE_REPORT_MD)

# Display concise outputs and confusion matrix
print("\n--- MFCC accuracy:", acc_mfcc)
if hubert_acc is not None:
    print("--- HuBERT sample accuracy:", hubert_acc)
print("\nConfusion matrix (MFCC):")
cm = confusion_matrix(yte, yp_mfcc, labels=present_labels)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=present_names, yticklabels=present_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("MFCC - Confusion Matrix")
plt.show()

print("\nReports and models saved. Review reports/ and models/ directories for artifacts.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# move into your project directory
%cd /content/drive/MyDrive/NLI_HuBERT_Project
!pwd
!ls -l

In [ ]:
!echo "Project base:"
!ls -la /content/drive/MyDrive/NLI_HuBERT_Project || true
!echo ""
!echo "Count .npz in features/hubert (old extractor):"
!find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert -type f -name "*.npz" 2>/dev/null | wc -l
!echo "List first 20 .npz (if any):"
!find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert -type f -name "*.npz" 2>/dev/null | head -n 20
!echo ""
!echo "Count .npy in features/hubert_npy (new extractor):"
!find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy -type f -name "*.npy" 2>/dev/null | wc -l
!echo "List first 20 .npy (if any):"
!find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy -type f -name "*.npy" 2>/dev/null | head -n 20

In [ ]:
%%bash
echo "Project base:"
ls -la /content/drive/MyDrive/NLI_HuBERT_Project || true
echo
echo "Count .npz in features/hubert (old extractor):"
find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert -type f -name "*.npz" 2>/dev/null | wc -l
echo "List first 20 .npz (if any):"
find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert -type f -name "*.npz" 2>/dev/null | head -n 20
echo
echo "Count .npy in features/hubert_npy (new extractor):"
find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy -type f -name "*.npy" 2>/dev/null | wc -l
echo "List first 20 .npy (if any):"
find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy -type f -name "*.npy" 2>/dev/null | head -n 20

In [ ]:
import os, numpy as np
from collections import Counter

folder = "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy"

files = [f for f in os.listdir(folder) if f.endswith('.npy')]
print("Total files:", len(files))

shapes = Counter()
nan_count = 0
example = None
for i,f in enumerate(files):
    arr = np.load(os.path.join(folder,f))
    shapes[arr.shape] += 1
    if np.isnan(arr).any():
        nan_count += 1
    if example is None:
        example = (f, arr.shape)
print("Unique shapes and counts:", shapes)
print("Files with NaNs:", nan_count)
print("Example file:", example)

In [ ]:
# Coverage check
from pathlib import Path
import pandas as pd
BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")
MANIFEST = BASE/"metadata.csv"
NPZ_DIR = BASE/"features"/"hubert"
NPY_DIR = BASE/"features"/"hubert_npy"
MFCC_DIR = BASE/"features"/"mfcc"
df = pd.read_csv(MANIFEST)
stems = [Path(s).stem for s in df['wav_path'].tolist()]
npz_stems = {p.stem for p in (NPZ_DIR.glob("*.npz") if NPZ_DIR.exists() else [])}
npy_stems = {p.stem for p in (NPY_DIR.glob("*.npy") if NPY_DIR.exists() else [])}
mfcc_stems = {p.stem for p in (MFCC_DIR.glob("*.npy") if MFCC_DIR.exists() else [])}
print("manifest rows:", len(stems))
print(".npz:", len(npz_stems), ".npy:", len(npy_stems), "mfcc:", len(mfcc_stems))
covered_any = sum(1 for s in stems if s in npz_stems or s in npy_stems)
missing = [s for s in stems if s not in npz_stems and s not in npy_stems]
print("Covered by any HuBERT:", covered_any, "Missing:", len(missing))
print("First 30 missing:", missing[:30])

In [ ]:
# Layer-by-layer on all available .npz (quick logistic eval per layer)
import numpy as np, pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")
NPZ_DIR = BASE/"features"/"hubert"
MANIFEST = BASE/"metadata.csv"
df = pd.read_csv(MANIFEST)

layer_feats = defaultdict(list)
labels = []; groups = []
n_layers = None
for _, r in df.iterrows():
    utt = Path(str(r['wav_path'])).stem
    p = NPZ_DIR / f"{utt}.npz"
    if not p.exists(): continue
    d = np.load(p)
    pooled = d['pooled'] if 'pooled' in d.files else d[d.files[0]]
    if pooled.ndim==1: pooled = pooled[None,:]
    if n_layers is None: n_layers = pooled.shape[0]
    if pooled.ndim==2 and pooled.shape[0]==n_layers:
        for L in range(n_layers): layer_feats[L].append(pooled[L])
        labels.append(str(r['label'])); groups.append(str(r.get('speaker_id', utt)))

print("Collected:", len(labels), "utterances; layers:", n_layers, "labels:", Counter(labels))
if len(labels)==0:
    print("No .npz pooled files found — skip layer analysis.")
else:
    le = LabelEncoder().fit(labels); y_enc = le.transform(labels)
    results = {}
    for L in sorted(layer_feats.keys()):
        X = np.vstack(layer_feats[L])
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        try:
            tr, te = next(gss.split(X, y_enc, groups))
        except Exception as e:
            results[L] = 0.0; continue
        scaler = StandardScaler().fit(X[tr]); Xtr, Xte = scaler.transform(X[tr]), scaler.transform(X[te])
        clf = LogisticRegression(max_iter=2000, solver='saga', multi_class='multinomial')
        clf.fit(Xtr, y_enc[tr])
        acc = accuracy_score(y_enc[te], clf.predict(Xte))
        results[L] = acc
        print(f"Layer {L}: acc {acc:.4f}")
    layers, accs = zip(*sorted(results.items()))
    plt.plot(layers, accs, marker='o'); plt.xlabel("Layer"); plt.ylabel("Acc"); plt.grid(True); plt.show()
    best_layer = max(results, key=results.get)
    print("Best layer (partial):", best_layer, "acc:", results[best_layer])

In [ ]:
# DIAGNOSTIC: check shapes, label distribution and splits for one layer
import numpy as np, pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit

BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")
NPZ_DIR = BASE/"features"/"hubert"
MANIFEST = BASE/"metadata.csv"
df = pd.read_csv(MANIFEST)

# load pooled for available .npz (same as earlier)
layer_feats = {}
labels=[]; groups=[]
n_layers=None
for _, r in df.iterrows():
    utt = Path(str(r['wav_path'])).stem
    p = NPZ_DIR / f"{utt}.npz"
    if not p.exists(): continue
    d = np.load(p)
    pooled = d['pooled'] if 'pooled' in d.files else d[d.files[0]]
    if pooled.ndim==1: pooled = pooled[None,:]
    if n_layers is None: n_layers = pooled.shape[0]
    labels.append(str(r['label'])); groups.append(str(r.get('speaker_id', utt)))
    for L in range(pooled.shape[0]):
        layer_feats.setdefault(L,[]).append(pooled[L])

print("Collected utterances:", len(labels), "layers:", n_layers)
print("Label counts (all):", Counter(labels))
print("Example groups counts:", Counter(groups).most_common(5))

# choose a layer
L = 0
X = np.vstack(layer_feats[L])
print("Layer", L, "X shape:", X.shape, "labels len:", len(labels), "groups len:", len(groups))
# speaker-wise split diagnostics
gss = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
ok = True
for i, (tr, te) in enumerate(gss.split(X, labels, groups)):
    tr_classes = Counter([labels[t] for t in tr])
    te_classes = Counter([labels[t] for t in te])
    print(f"Fold {i}: train size {len(tr)} test size {len(te)} | train classes {len(tr_classes)} test classes {len(te_classes)}")
    print(" train class counts sample:", tr_classes.most_common(3))
    print(" test class counts sample:", te_classes.most_common(3))
    # coverage check
    if len(set(tr_classes.keys()).intersection(set(te_classes.keys())))==0:
        print(" >>> WARNING: no class overlap between train and test in this fold!")
        ok=False

print("Split sanity ok?", ok)

In [ ]:
# FIXED robust layer-by-layer analysis with correct speaker-group detection
import numpy as np, pandas as pd, json
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")
NPZ_DIR = BASE/"features"/"hubert"
MANIFEST = BASE/"metadata.csv"
MIN_UTTS = 4
TEST_SIZE = 0.2
RANDOM_STATE = 42

df = pd.read_csv(MANIFEST)
print("Manifest columns:", df.columns.tolist(), "rows:", len(df))

#  robust column detection
candidate_utt = ["utt_id","utt","id","filename","file","wav","wav_path","path","audio_path","audio"]
candidate_spk = ["speaker_id","speaker","spk","speakerID","spk_id","speaker-id"]
candidate_lab = ["label","lang","language","target"]

utt_col = next((c for c in candidate_utt if c in df.columns), None)
spk_col = next((c for c in candidate_spk if c in df.columns), None)
lab_col = next((c for c in candidate_lab if c in df.columns), None)
print("Detected -> utt_col:", utt_col, "spk_col:", spk_col, "lab_col:", lab_col)
if lab_col is None:
    raise KeyError("No label column found in manifest (update candidate_lab list)")

# helper to get utt stem
def utt_stem_from_row(r):
    if utt_col and pd.notna(r.get(utt_col)):
        return Path(str(r.get(utt_col))).stem
    # fallback try wav_path or path fields
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(r.get(c)):
            return Path(str(r.get(c))).stem
    return str(r.name)

# helper to derive speaker id robustly
def spk_from_row(r):
    # prefer explicit speaker column if present and seems not equal to label
    if spk_col and spk_col in df.columns and pd.notna(r.get(spk_col)):
        cand = str(r.get(spk_col))
        # check cand not equal to label for obvious mis-assignments
        if cand.lower() != str(r.get(lab_col)).lower():
            return cand
    # fallback: derive from wav path parent directory
    for c in [utt_col, "wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(r.get(c)):
            p = Path(str(r.get(c)))
            if p.parent != Path("") and p.parent.name != "":
                return p.parent.name
    # final fallback: use utterance stem (unique per utterance)
    return utt_stem_from_row(r)

# Build layer_feats and correct labels/groups
layer_feats = defaultdict(list)
labels = []; groups = []
n_layers = None
skipped = 0
for _, r in df.iterrows():
    utt_stem = utt_stem_from_row(r)
    p = NPZ_DIR / f"{utt_stem}.npz"
    if not p.exists():
        skipped += 1
        continue
    d = np.load(p)
    # get pooled or first array
    pooled = d['pooled'] if 'pooled' in d.files else d[d.files[0]]
    if pooled is None:
        skipped += 1
        continue
    if pooled.ndim == 1:
        pooled = pooled[None,:]
    if n_layers is None:
        n_layers = pooled.shape[0]
    # append per-layer vectors
    for L in range(pooled.shape[0]):
        layer_feats[L].append(pooled[L])
    labels.append(str(r[lab_col]))
    groups.append(spk_from_row(r))

print("Collected pooled for utts:", len(labels), "missing/skipped:", skipped, "detected n_layers:", n_layers)
print("Sample groups (top 10):", Counter(groups).most_common(10))
print("Label counts:", Counter(labels))

# Filter tiny classes
counts = Counter(labels)
keep_labels = {lab for lab,c in counts.items() if c >= MIN_UTTS}
if len(keep_labels) < len(counts):
    print("Filtering out classes with <", MIN_UTTS, "utterances. Kept:", keep_labels)
    mask = [lab in keep_labels for lab in labels]
    labels = [lab for i,lab in enumerate(labels) if mask[i]]
    groups = [g for i,g in enumerate(groups) if mask[i]]
    for L in list(layer_feats.keys()):
        layer_feats[L] = [layer_feats[L][i] for i in range(len(layer_feats[L])) if mask[i]]

print("After filtering, examples:", len(labels), "classes:", Counter(labels))

# label encode
le = LabelEncoder().fit(labels)
y_enc = le.transform(labels)

# Robust per-layer eval
results = {}
for L in sorted(layer_feats.keys()):
    X = np.vstack(layer_feats[L])
    # Try speaker-wise split
    try:
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        tr, te = next(gss.split(X, y_enc, groups))
        tr_cls = set(y_enc[tr]); te_cls = set(y_enc[te])
        # require some class overlap (>=2 classes in both)
        if len(tr_cls & te_cls) < 1 or len(tr_cls) < 2 or len(te_cls) < 2:
            raise ValueError("Degenerate speaker-wise split (insufficient overlap) -> fallback")
        mode = "group"
    except Exception as e:
        # fallback - stratified utterance-level split
        tr, te = train_test_split(np.arange(len(X)), test_size=TEST_SIZE, stratify=y_enc, random_state=RANDOM_STATE)
        mode = "strat"
    Xtr, Xte = X[tr], X[te]; ytr, yte = y_enc[tr], y_enc[te]
    print(f"Layer {L:02d} split mode={mode} | train {len(tr)} ({len(set(ytr))} classes) test {len(te)} ({len(set(yte))} classes)")
    scaler = StandardScaler().fit(Xtr); Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
    clf = LogisticRegression(max_iter=2000, solver='saga', multi_class='multinomial')
    clf.fit(Xtr_s, ytr)
    acc = accuracy_score(yte, clf.predict(Xte_s))
    results[L] = acc
    print(f" -> Layer {L:02d} acc: {acc:.4f}")

# plot + save
import matplotlib.pyplot as plt
layers, accs = zip(*sorted(results.items()))
plt.figure(figsize=(9,4)); plt.plot(layers, accs, marker='o'); plt.xlabel("HuBERT layer"); plt.ylabel("acc"); plt.grid(True); plt.show()
best_layer = max(results, key=results.get)
print("Best layer:", best_layer, "acc:", results[best_layer])
with open(BASE/"reports"/"layerwise_results_fixed_groups.json","w") as f:
    json.dump(results,f)
print("Saved ->", BASE/"reports"/"layerwise_results_fixed_groups.json")

In [ ]:
# Attempt to auto-extract speaker ids (heuristics) and re-run speaker-wise layer analysis
import re, json
import numpy as np, pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")
NPZ_DIR = BASE/"features"/"hubert"
MANIFEST = BASE/"metadata.csv"
MIN_UTTS = 4
TEST_SIZE = 0.2
RANDOM_STATE = 42

df = pd.read_csv(MANIFEST)
print("Manifest lines:", len(df))

def utt_stem_from_row(r, utt_col="wav_path"):
    if utt_col in r and pd.notna(r[utt_col]):
        return Path(str(r[utt_col])).stem
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in r and pd.notna(r[c]):
            return Path(str(r[c])).stem
    return str(r.name)

# Heuristics for speaker extraction from filename
def extract_speaker_from_stem(stem):
    s = stem
    # common patterns: "lang_speaker_03_10" => capture lang_speaker_03
    m = re.search(r'([A-Za-z]+[-]speaker[-]?\d+)', s, flags=re.I)
    if m: return m.group(1).lower()
    # "lang_speaker03" or "lang-speaker03"
    m = re.search(r'([A-Za-z]+[-]?speaker[-]?\d+)', s, flags=re.I)
    if m: return m.group(1).lower()
    # "speaker_03_..." -> speaker_03
    m = re.search(r'(speaker[_-]?\d+)', s, flags=re.I)
    if m: return m.group(1).lower()
    # patterns like "gujrat_02_14" where second token is speaker id -> gujrat_02
    tokens = re.split(r'[_\-\. ]+', s)
    if len(tokens) >= 2 and tokens[1].isdigit():
        return f"{tokens[0]}_{tokens[1]}".lower()
    # fallback: take first two tokens if many tokens
    if len(tokens) >= 2:
        return f"{tokens[0]}_{tokens[1]}".lower()
    return s.lower()

# Build mapping for many stems and show examples
stems = [utt_stem_from_row(r) for _,r in df.iterrows()]
unique_stems = sorted(set(stems))
sample = unique_stems[:200]  # sample first 200 to examine
mapping = {stem: extract_speaker_from_stem(stem) for stem in sample}

print("Example extracted speakers from filenames (first 30):")
for i, (k,v) in enumerate(mapping.items()):
    if i>=30: break
    print(k, "->", v)

# Now build dataset using NPZ pooled and derived speaker ids
layer_feats = defaultdict(list)
labels = []; groups = []
n_layers = None
skipped = 0
for _, r in df.iterrows():
    stem = utt_stem_from_row(r)
    p = NPZ_DIR / f"{stem}.npz"
    if not p.exists():
        skipped += 1
        continue
    d = np.load(p)
    pooled = d['pooled'] if 'pooled' in d.files else d[d.files[0]]
    if pooled is None:
        skipped += 1; continue
    if pooled.ndim == 1: pooled = pooled[None,:]
    if n_layers is None: n_layers = pooled.shape[0]
    for L in range(pooled.shape[0]):
        layer_feats[L].append(pooled[L])
    labels.append(str(r['label']))
    groups.append(extract_speaker_from_stem(stem))

print("Collected pooled:", len(labels), "skipped:", skipped, "layers:", n_layers)
print("Example group counts top10:", Counter(groups).most_common(10))
print("Label counts:", Counter(labels))

# Filter tiny classes
keep = {lab for lab,c in Counter(labels).items() if c>=MIN_UTTS}
mask = [lab in keep for lab in labels]
if sum(mask)!=len(labels):
    print("Filtering tiny classes (<{}): removed {} examples".format(MIN_UTTS, len(labels)-sum(mask)))
    labels = [lab for i,lab in enumerate(labels) if mask[i]]
    groups = [g for i,g in enumerate(groups) if mask[i]]
    for L in list(layer_feats.keys()):
        layer_feats[L] = [layer_feats[L][i] for i in range(len(layer_feats[L])) if mask[i]]

print("After filter, examples:", len(labels), "classes:", Counter(labels))
le = LabelEncoder().fit(labels)
y_enc = le.transform(labels)

# Try to perform a speaker-wise split and check for overlap
def speaker_split_ok(groups, y_enc, test_size=TEST_SIZE):
    try:
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_STATE)
        tr, te = next(gss.split(np.zeros(len(groups)), y_enc, groups))
        tr_cls, te_cls = set(y_enc[tr]), set(y_enc[te])
        overlap = len(tr_cls & te_cls)
        return overlap >= 1, tr, te
    except Exception:
        return False, None, None

ok, tr_idx, te_idx = speaker_split_ok(groups, y_enc)
print("Speaker-wise split possible?", ok)
if not ok:
    print("Speaker-wise split not possible with derived speakers -> we should inspect speaker ids or use stratified fallback (but will cause leakage).")
else:
    print("Speaker-wise split OK. Running per-layer evaluation (speaker-wise).")
    results = {}
    from sklearn.preprocessing import StandardScaler
    for L in sorted(layer_feats.keys()):
        X = np.vstack(layer_feats[L])
        Xtr, Xte = X[tr_idx], X[te_idx]
        ytr, yte = y_enc[tr_idx], y_enc[te_idx]
        print(f"Layer {L}: train {Xtr.shape[0]} test {Xte.shape[0]} | train class count {len(set(ytr))} test class count {len(set(yte))}")
        scaler = StandardScaler().fit(Xtr); Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
        clf = LogisticRegression(max_iter=2000, solver='saga', multi_class='multinomial')
        clf.fit(Xtr_s, ytr)
        acc = accuracy_score(yte, clf.predict(Xte_s))
        results[L] = acc
        print(f" -> Layer {L} acc: {acc:.4f}")
    # plot and save
    import matplotlib.pyplot as plt
    layers, accs = zip(*sorted(results.items()))
    plt.figure(figsize=(9,4)); plt.plot(layers, accs, marker='o'); plt.xlabel("layer"); plt.ylabel("acc"); plt.grid(True); plt.show()
    best_layer = max(results, key=results.get)
    print("Best layer (speaker-wise):", best_layer, "acc:", results[best_layer])
    with open(BASE/"reports"/"layerwise_results_auto_speakers.json","w") as f:
        json.dump(results,f)
    print("Saved ->", BASE/"reports"/"layerwise_results_auto_speakers.json")

In [ ]:
# Train HuBERT best-layer RandomForest (safe, speaker-wise)
import numpy as np, joblib, json
from pathlib import Path
from collections import Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")
MANIFEST = BASE/"metadata.csv"
NPZ_DIR = BASE/"features"/"hubert"          # per-layer .npz (540 subset)
NPY_DIR = BASE/"features"/"hubert_npy"      # full pooled .npy (8116)
MODEL_DIR = BASE/"models"
REPORTS_DIR = BASE/"reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

best_layer = 8          # from your layerwise analysis
MIN_UTTS = 3            # drop classes with fewer than this many utts
TEST_SIZE = 0.20
RANDOM_STATE = 42
USE_PCA = True
PCA_MAX_COMP = 256

# Load manifest
import pandas as pd
df = pd.read_csv(MANIFEST)
print("Manifest rows:", len(df))

# Build dataset (npz -> layer vector; else npy)
X_list, y_list, groups = [], [], []
missing = 0
for _, r in df.iterrows():
    utt = Path(str(r['wav_path'])).stem
    p_npz = NPZ_DIR / f"{utt}.npz"
    p_npy = NPY_DIR / f"{utt}.npy"
    vec = None
    if p_npz.exists():
        d = np.load(p_npz)
        pooled = d['pooled'] if 'pooled' in d.files else d[d.files[0]]
        if pooled.ndim==2 and pooled.shape[0] > best_layer:
            vec = pooled[best_layer]
        else:
            vec = pooled.mean(axis=0) if pooled.ndim==2 else pooled
    # elif p_npy.exists():
        # vec = np.load(p_npy)
    else:
        missing += 1
        continue
    X_list.append(vec)
    y_list.append(str(r['label']))
    groups.append(str(r.get('speaker_id', utt)))

print(f"Built raw dataset: {len(X_list)} examples, missing {missing}")

if len(X_list) == 0:
    raise RuntimeError("No features found (check NPZ/NPY paths).")

X = np.vstack(X_list)
y = np.array(y_list)
groups = np.array(groups)
print("X shape:", X.shape, "labels:", Counter(y))

# Filter tiny classes
cnt = Counter(y)
keep = {lab for lab,c in cnt.items() if c >= MIN_UTTS}
if len(keep) < len(cnt):
    print("Filtering small classes (<{}). Keeping: {}".format(MIN_UTTS, sorted(keep)))
    mask = np.array([lab in keep for lab in y])
    X, y, groups = X[mask], y[mask], groups[mask]
    print("After filter X shape:", X.shape, "labels:", Counter(y))

# Encode labels & speaker-wise split
le = LabelEncoder().fit(y)
y_enc = le.transform(y)

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(X, y_enc, groups))
Xtr, Xte = X[tr_idx], X[te_idx]
ytr, yte = y_enc[tr_idx], y_enc[te_idx]
groups_tr, groups_te = groups[tr_idx], groups[te_idx]
print("Speaker-wise split -> train:", Xtr.shape, "test:", Xte.shape)
print("Train speakers:", len(set(groups_tr)), "Test speakers:", len(set(groups_te)))
print("Train label counts:", Counter(ytr), "Test label counts:", Counter(yte))

# Scale
scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)

# PCA (safely)
if USE_PCA:
    max_comp = min(PCA_MAX_COMP, Xtr_s.shape[1], Xtr_s.shape[0]-1)
    if max_comp <= 0:
        print("Skipping PCA due to insufficient samples/features.")
        Xtr_p, Xte_p = Xtr_s, Xte_s
        pca = None
    else:
        pca = PCA(n_components=max_comp, random_state=RANDOM_STATE).fit(Xtr_s)
        Xtr_p, Xte_p = pca.transform(Xtr_s), pca.transform(Xte_s)
        print("PCA applied. Components:", max_comp)
else:
    pca = None
    Xtr_p, Xte_p = Xtr_s, Xte_s

# Train RF
clf = RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
print("Training RandomForest...")
clf.fit(Xtr_p, ytr)

# Eval
yp = clf.predict(Xte_p)
acc = accuracy_score(yte, yp)
print("Test accuracy (speaker-wise):", acc)
print(classification_report(yte, yp, target_names=le.classes_, zero_division=0))

# Confusion matrix
cm = confusion_matrix(yte, yp)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"HuBERT best-layer {best_layer} RF (acc={acc:.3f})")
plt.show()

# Save artifacts
artifacts = {
    "clf": clf,
    "scaler": scaler,
    "pca": pca,
    "label_encoder": le,
    "best_layer": best_layer,
    "feature_counts": dict(cnt)
}
joblib.dump(artifacts, MODEL_DIR/f"hubert_bestlayer_rf_layer{best_layer}.joblib")
with open(REPORTS_DIR/f"hubert_bestlayer_results_layer{best_layer}.json","w") as f:
    json.dump({"accuracy": float(acc), "classes": le.classes_.tolist(), "best_layer": int(best_layer)}, f)

print("Saved model and reports to:", MODEL_DIR, REPORTS_DIR)

In [ ]:
#  best-layer RF training

import os, json, warnings
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

#  CONFIG
MANIFEST = "metadata.csv"  # path to manifest CSV (must contain wav_path/speaker_id/label or similar)
HUBERT_DIR = "features/hubert"    # existing .npz (pooled per-layer)
HUBERT_NPY_DIR = "features/hubert_npy"  # optional: .npy single-vector embeddings (if available)
MODEL_DIR = Path("models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MIN_UTTS_KEEP = 4       # remove labels with fewer than this many utterances
TEST_SIZE = 0.20
RANDOM_STATE = 42
PCA_MAX = 256           # max PCA components
RF_ESTIMATORS = 300


# Helper: detect columns
if not Path(MANIFEST).exists():
    raise FileNotFoundError(f"Manifest not found at {MANIFEST}")
df = pd.read_csv(MANIFEST)
print("Manifest lines:", len(df))
print("Manifest columns:", list(df.columns))

# Candidate column names
candidate_utt_cols = ["utt_id","utt","id","filename","file","wav","wav_path","path","audio_path","audio"]
candidate_spk_cols = ["speaker_id","speaker","spk","speakerID","spk_id","speakerid"]
candidate_label_cols = ["label","lang","language","target"]

utt_col = next((c for c in candidate_utt_cols if c in df.columns), None)
spk_col = next((c for c in candidate_spk_cols if c in df.columns), None)
lab_col = next((c for c in candidate_label_cols if c in df.columns), None)

if lab_col is None:
    raise KeyError("No label column found in manifest. Edit candidate_label_cols or the manifest.")

print("Detected -> utt_col:", utt_col, "spk_col:", spk_col, "lab_col:", lab_col)

# Helpers to extract utt stem and speaker robustly
def utt_stem_from_row(row):
    if utt_col and pd.notna(row.get(utt_col)):
        val = str(row.get(utt_col))
        # if it's a path, return stem; else normalize spaces
        p = Path(val)
        return p.stem if p.suffix else val
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).stem
    # fallback to index
    return str(row.name)

def spk_from_row(row):
    if spk_col and pd.notna(row.get(spk_col)):
        return str(row.get(spk_col))
    # fallback to parent folder name if path present
    for c in ["wav_path","path","audio_path","file","filename"]:
        if c in df.columns and pd.notna(row.get(c)):
            return Path(str(row.get(c))).parent.name
    return "unknown"

# Index available HuBERT files (.npz preferred, .npy alternative)
hub_dir = Path(HUBERT_DIR)
npy_dir = Path(HUBERT_NPY_DIR)
hubert_index_npz = {p.stem: p for p in hub_dir.rglob("*.npz")} if hub_dir.exists() else {}
hubert_index_npy = {p.stem: p for p in npy_dir.rglob("*.npy")} if npy_dir.exists() else {}
print("Indexed .npz files:", len(hubert_index_npz), "Indexed .npy files:", len(hubert_index_npy))

# Collect pooled arrays (per-layer) for as many utts as available.
# For .npz: prefer pooled (n_layers, dim). For .npy: treat as single pooled layer (1, dim).
utt_to_pooled = {}
collected_utts = []
skipped = 0
n_layers_detected = None

for _, row in df.iterrows():
    utt = utt_stem_from_row(row)
    # prefer .npz
    p = hubert_index_npz.get(utt) or hubert_index_npz.get(utt.replace(" ", "_"))
    used_format = None
    if p:
        try:
            data = np.load(p, allow_pickle=True)
        except Exception as e:
            skipped += 1
            continue
        # common key 'pooled'
        if "pooled" in data:
            pooled = data["pooled"]
        elif "hidden_states" in data:
            # hidden_states: tuple/list of layers, each (seq_len, dim) -> mean across seq
            pooled = np.array([h.mean(axis=0) for h in data["hidden_states"]])
        else:
            # pick first 2D array in file
            arrs = [data[k] for k in data.files]
            two_d = next((a for a in arrs if getattr(a,"ndim",0)==2), None)
            if two_d is not None:
                pooled = two_d
            elif len(arrs)==1 and arrs[0].ndim==1:
                pooled = arrs[0][None,:]
            else:
                skipped += 1
                continue
        used_format = ".npz"
    else:
        # try .npy (single pooled vector)
        q = hubert_index_npy.get(utt) or hubert_index_npy.get(utt.replace(" ", "_"))
        if q:
            try:
                arr = np.load(q)
            except Exception as e:
                skipped += 1
                continue
            if arr.ndim == 1:
                pooled = arr[None,:]
            elif arr.ndim == 2:
                pooled = arr  # interpret as layers or pooled vectors; we'll handle shape later
            else:
                skipped += 1
                continue
            used_format = ".npy"
        else:
            skipped += 1
            continue

    # register
    if n_layers_detected is None:
        n_layers_detected = int(pooled.shape[0])
    # only keep if pooled has consistent number of layers (or is single-layer)
    if pooled.ndim==2 and pooled.shape[0] == n_layers_detected:
        utt_to_pooled[utt] = pooled
        collected_utts.append(utt)
    elif pooled.ndim==2 and pooled.shape[0] == 1:
        # single-layer npy fits as 1 layer
        # if n_layers_detected is greater, we still accept (will average later or use layer-mean)
        utt_to_pooled[utt] = pooled
        collected_utts.append(utt)
    else:
        # inconsistent layer shapes - accept but we will handle later
        utt_to_pooled[utt] = pooled
        collected_utts.append(utt)

print(f"Collected pooled: {len(collected_utts)} skipped: {skipped} detected n_layers: {n_layers_detected}")

if len(collected_utts) == 0:
    raise RuntimeError("No HuBERT pooled features found. Run extractor and re-run this cell.")

# Quick per-layer scan to pick a good layer (logreg on pooled layer vectors) --
# uses only collected_utts to speed up selection.
print("Running quick per-layer scan on available pooled files...")

# Build labels & groups aligned with collected_utts
labels = []
groups = []
for u in collected_utts:
    row = df[df.apply(lambda r: utt_stem_from_row(r)==u, axis=1)]
    if row.shape[0] == 0:
        labels.append("unknown")
        groups.append("unknown")
    else:
        labels.append(str(row.iloc[0][lab_col]))
        groups.append(spk_from_row(row.iloc[0]))
labels = np.array(labels)
groups = np.array(groups)
print("Label counts (all):", Counter(labels))
unique_labels = sorted(list(set(labels)))

# Determine number of layers to test:
# if some pooled arrays are single-layer (shape[0]==1) and n_layers_detected is None, use 1
if n_layers_detected is None:
    # fallback to 1
    n_layers = 1
else:
    n_layers = int(n_layers_detected)

# gather per-layer vectors (if a file has only 1 layer, we will broadcast that vector for each layer slot)
layer_feats = defaultdict(list)
for u in collected_utts:
    pooled = utt_to_pooled[u]
    if pooled.ndim == 1:
        pooled = pooled[None,:]
    # if pooled has fewer layers than n_layers, we repeat/mean to fill
    if pooled.shape[0] == n_layers:
        for l in range(n_layers):
            layer_feats[l].append(pooled[l])
    elif pooled.shape[0] == 1 and n_layers > 1:
        # broadcast single pooled vector to all layers (not ideal but allows using npy files)
        for l in range(n_layers):
            layer_feats[l].append(pooled[0])
    else:
        # handle unexpected shape by mean across layers
        meanv = pooled.mean(axis=0)
        for l in range(n_layers):
            layer_feats[l].append(meanv)

# encode labels for quick scan
le_scan = LabelEncoder().fit(labels)
y_scan = le_scan.transform(labels)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

layer_scores = {}
for l in sorted(layer_feats.keys()):
    Xl = np.vstack(layer_feats[l])
    # if not enough diversity, skip
    if Xl.shape[0] < 10 or len(np.unique(y_scan)) < 2:
        layer_scores[l] = 0.0
        continue
    # speaker-wise split if possible
    try:
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        tr, te = next(gss.split(Xl, y_scan, groups))
        Xtr, Xte = Xl[tr], Xl[te]; ytr, yte = y_scan[tr], y_scan[te]
    except Exception:
        Xtr, Xte, ytr, yte = train_test_split(Xl, y_scan, test_size=TEST_SIZE, stratify=y_scan, random_state=RANDOM_STATE)
    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
    clf = LogisticRegression(max_iter=2000, solver="saga", multi_class="multinomial")
    try:
        clf.fit(Xtr_s, ytr)
        acc = accuracy_score(yte, clf.predict(Xte_s))
    except Exception:
        acc = 0.0
    layer_scores[l] = float(acc)
    print(f"Layer {l:02d} -> quick-acc: {acc:.4f}")

# pick best layer (highest quick accuracy)
best_layer = max(layer_scores, key=layer_scores.get)
print("Picked best layer (quick scan):", best_layer, "acc:", layer_scores[best_layer])

# Save quick layer results for reproducibility
with open(REPORTS_DIR / "layerwise_results_quick.json", "w") as f:
    json.dump(layer_scores, f)

#  Build full dataset using chosen layer
X_list, y_list, groups_list, utts_list = [], [], [], []
for _, row in df.iterrows():
    utt = utt_stem_from_row(row)
    pooled = utt_to_pooled.get(utt)
    if pooled is None:
        # try normalized names
        pooled = utt_to_pooled.get(utt.replace(" ", "_"))
    if pooled is None:
        continue
    if pooled.ndim == 1:
        # single vector -> use it
        vec = pooled
    elif pooled.ndim == 2:
        # if has best_layer index, use that, else mean across layers
        if pooled.shape[0] > best_layer:
            vec = pooled[best_layer]
        else:
            vec = pooled.mean(axis=0)
    else:
        continue
    X_list.append(vec)
    y_list.append(str(row[lab_col]))
    groups_list.append(spk_from_row(row))
    utts_list.append(utt)

if len(X_list) == 0:
    raise RuntimeError("No dataset built for HuBERT best-layer. Check pooled files and manifest matching.")

X = np.vstack(X_list)
y = np.array(y_list)
groups = np.array(groups_list)
print("Built dataset:", X.shape, "labels:", Counter(y))

# Filter tiny classes
cnts = Counter(y)
keep_labels = {lab for lab,c in cnts.items() if c >= MIN_UTTS_KEEP}
print("Original class counts:", cnts)
print("Keeping labels (>= {} utts): {}".format(MIN_UTTS_KEEP, sorted(keep_labels)))

Xf, yf, gf = [], [], []
for xi, yi, gi in zip(X, y, groups):
    if yi in keep_labels:
        Xf.append(xi); yf.append(yi); gf.append(gi)
Xf = np.vstack(Xf); yf = np.array(yf); gf = np.array(gf)
print("After filtering dataset shape:", Xf.shape, "labels:", Counter(yf))

if Xf.shape[0] < 20:
    raise RuntimeError("Too few samples after filtering. Lower MIN_UTTS_KEEP or add more data.")

# Encode labels
le = LabelEncoder().fit(yf)
y_enc = le.transform(yf)
print("Present classes:", list(le.classes_))

# speaker-wise split with fallback to stratified utterance split (if speaker-split cannot preserve classes)
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
try:
    tr, te = next(gss.split(Xf, y_enc, gf))
    # check test contains at least 2 classes
    if len(np.unique(y_enc[te])) < 2:
        raise ValueError("speaker-wise split produced too few test classes, falling back to stratified utterance split")
    split_mode = "speaker-wise"
except Exception as e:
    print("Warning:", e)
    # fallback: stratified utterance-level split (not speaker-safe)
    tr, te = train_test_split(np.arange(len(Xf)), test_size=TEST_SIZE, stratify=y_enc, random_state=RANDOM_STATE)
    split_mode = "stratified_utterance"
print("Split mode used:", split_mode)
Xtr, Xte = Xf[tr], Xf[te]; ytr, yte = y_enc[tr], y_enc[te]
print("Train/Test sizes:", Xtr.shape, Xte.shape, "train labels:", Counter(ytr), "test labels:", Counter(yte))

# Scale
scaler = StandardScaler().fit(Xtr)
Xtr_s = scaler.transform(Xtr); Xte_s = scaler.transform(Xte)

# PCA: safe n_components
max_components = min(PCA_MAX, Xtr_s.shape[1], Xtr_s.shape[0]-1)
if max_components <= 0:
    print("Skipping PCA due to small sample size.")
    Xtr_p, Xte_p = Xtr_s, Xte_s
    pca = None
else:
    pca = PCA(n_components=max_components, random_state=RANDOM_STATE).fit(Xtr_s)
    Xtr_p, Xte_p = pca.transform(Xtr_s), pca.transform(Xte_s)
    print("PCA applied, components:", max_components)

# Train RandomForest
clf = RandomForestClassifier(n_estimators=RF_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")
clf.fit(Xtr_p, ytr)
yp = clf.predict(Xte_p)
acc = accuracy_score(yte, yp)
print(f"HuBERT best-layer ({best_layer}) speaker-wise accuracy: {acc:.4f}")

# Safe classification report (always list all classes present in label encoder)
labels_idx = list(range(len(le.classes_)))
print("Classification report (present labels only):")
print(classification_report(yte, yp, labels=labels_idx, target_names=le.classes_, zero_division=0))

# Confusion matrix (with labels param so shape is consistent)
cm = confusion_matrix(yte, yp, labels=labels_idx)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"HuBERT layer {best_layer} - Confusion Matrix")
plt.show()

# Save artifacts
artifacts = {
    "clf": clf,
    "scaler": scaler,
    "pca": pca,
    "label_encoder": le,
    "best_layer": int(best_layer),
    "split_mode": split_mode,
    "present_classes": le.classes_.tolist()
}
joblib.dump(artifacts, MODEL_DIR / f"hubert_bestlayer_rf_layer{best_layer}.joblib")
with open(REPORTS_DIR / f"hubert_bestlayer_report_layer{best_layer}.json", "w") as f:
    json.dump({
        "accuracy": float(acc),
        "best_layer": int(best_layer),
        "class_counts": {k:int(v) for k,v in Counter(yf).items()},
        "split_mode": split_mode,
        "labels": le.classes_.tolist()
    }, f, indent=2)
print("Saved model artifacts and report to:", MODEL_DIR, REPORTS_DIR)

In [ ]:
# ✅ RandomForest Hyperparameter Tuning (works with or without SMOTE)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
import joblib, os
from collections import Counter


# Auto-detect which training data exists

if "X_tr_bal" in locals() and "y_tr_bal" in locals():
    print("📦 Using SMOTE-balanced training data (X_tr_bal)")
    X_train_tune, y_train_tune = X_tr_bal, y_tr_bal
else:
    print("📦 Using original training data (X_tr)")
    X_train_tune, y_train_tune = X_tr_p, y_tr


# Define hyperparameter grid

param_grid = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [20, 40, 60, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": ["balanced"]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

grid = GridSearchCV(
    rf_base,
    param_grid,
    cv=cv,
    n_jobs=-1,
    scoring="accuracy",
    verbose=2
)

print(f"🔍 Starting Grid Search on {X_train_tune.shape[0]} samples, {X_train_tune.shape[1]} features")
grid.fit(X_train_tune, y_train_tune)


# Show results

print("\n✅ Best Parameters:", grid.best_params_)
print(f"🏁 Best CV Accuracy: {grid.best_score_:.4f}")

best_rf = grid.best_estimator_
best_rf.fit(X_train_tune, y_train_tune)

print("\n📈 Train accuracy:", best_rf.score(X_train_tune, y_train_tune))
print("🧪 Test accuracy:", best_rf.score(X_te_p, y_te))

# Save tuned model

os.makedirs("models", exist_ok=True)
joblib.dump(best_rf, "models/rf_tuned_best.joblib")
print("\n💾 Tuned model saved as: models/rf_tuned_best.joblib")

---

## Production Implementation: Modular Scripts

Due to notebook execution challenges (memory constraints, kernel instability), the production system was refactored into modular Python scripts in `scripts/`.

This notebook contains the exploratory analysis and initial experiments. **For reproducible results and deployment, refer to:**

### Core Scripts
- `scripts/train_final_model.py` - Final model training (99.73% accuracy)
- `scripts/predict_backend.py` - Unified inference backend
- `scripts/persistent_hubert.py` - Persistent HuBERT loader
- `app.py` - Gradio web interface

### Testing
```bash
# Run prediction test
python scripts/test_predict_backend.py "data/raw/karnataka/Karnataka_speaker_02_1 (744).wav"

# Launch web interface
python app.py
```

### Documentation
- `README.md` - Complete usage guide
- `SUBMISSION_CHECKLIST.md` - Project status and readiness
- `KNOWN_ISSUES.md` - Edge cases and solutions